# Text2Cypher Toolkit: Edge (CPU) Evaluation on Kaggle

Companion to `final-eval-kaggle.ipynb`. It measures edge (CPU) deployment and adds the analyses the GPU evaluation does not produce. Every result is written as structured tables for the thesis.

### What it measures
- **Export:** each final adapter is merged into the full-precision base model, converted to an f16 GGUF and quantised with a pinned llama.cpp build. Adapter verification, file sizes and SHA-256 hashes are recorded.
- **Throughput:** `llama-bench` prompt-processing (ppN) and generation (tgN) speed per model, quantisation and thread count.
- **Footprint:** `llama-server` load time, resident memory after load, peak resident memory and the first (cold) request.
- **Accuracy on the CPU:** GGUF predictions for every run (Q4_K_M), the cross-schema condition and extra quantisations if time allows, on a seeded subset of the 500 rows used by `final-eval`. Prompts are built by the same code as `app.py predict`; scoring uses `app.py evaluate --execute --validate-syntax`.
- **Fidelity:** paired comparison of every CPU job with the matching GPU (4-bit HF) job on identical rows, including the share of identical predictions.
- **Per-query latency:** prefill and decode time from the server's own timings.
- **Tokenizer parity:** prompt tokens counted by the GGUF tokenizer vs the training tokenizer (catches a doubled or missing BOS).
- **Supplementary analyses of the GPU reference jobs:**
  - metrics without the near-saturated `neo4jLabs_functional_cypher` source;
  - whether the exact-match filter keeps every element the gold query needs;
  - schema hallucination;
  - execution matches where the gold query returns no rows;
  - cross-model agreement;
  - sample composition.

### Setup (per Kaggle account)
1. **Settings > Accelerator > None** (CPU session; no GPU quota is used). **Settings > Internet > On.**
2. **Add-ons > Secrets:** `HF_TOKEN` with read access to the checkpoint repo and write access for uploads.
3. Pin `GIT_REF` (and optionally `LLAMA_CPP_REF`) in the configuration cell.

### Running and resuming
- **Save Version > Save & Run All (Commit).**
- Each finished job, benchmark and export record is uploaded to `edge-eval/<edge_tag>/` on the Hub in one commit.
- If the session ends early, commit a new version. Finished work is reused when the code commit, llama.cpp commit, CPU model and row set all match. Only runs with unfinished work are exported again.
- The first session records its llama.cpp commit on the Hub. Later sessions with a blank `LLAMA_CPP_REF` build that same commit.

### Caveats
- **Hardware:** timings come from the Kaggle CPU recorded in the manifest (`lscpu`), not from a specific edge device. The thread sweep shows how speed scales with fewer cores.
- **Quantisation:** adapters were trained against an NF4 base and are merged into the full-precision base before GGUF quantisation. The GPU vs CPU comparison measures both changes together.

In [ ]:
# 1. Edge evaluation configuration (edit, then Save Version > Save & Run All)
import time

PIPELINE_T0 = time.time()  # Session time budget is measured from here

HUB_REPO = '2l3c34o1/text2cypher-checkpoints'
GITHUB_REPO = 'https://github.com/Cxmrykk/capstone.git'
GIT_REF = ''                  # Pin for thesis runs (tag or commit); blank is warned and flagged
LLAMA_CPP_REPO = 'https://github.com/ggml-org/llama.cpp'
LLAMA_CPP_REF = ''            # Tag or commit. Blank: the commit recorded by the first session, else latest

REFERENCE_EVAL_TAG = 'test-n500-seed3407-r030'   # GPU jobs written by final-eval-kaggle (history/030.md)
EDGE_TAG_SUFFIX = 'e032'                         # Hub namespace: edge-eval/<edge_tag>/
REUSE_SAME_COMMIT_ONLY = True # Reuse Hub work only from this code commit (llama.cpp commit, CPU and rows always must match)
SKIP_COMPLETED_JOBS = True

# Experiments trained on Kaggle: (model, schema variant, config).
EXPERIMENTS = [
    ('gemma-4-e2b', 'enhanced', 'configs/gemma-4-e2b.yaml'),
    ('gemma-4-e2b', 'pruned', 'configs/gemma-4-e2b-pruned.yaml'),
    ('qwen3.5-2b', 'enhanced', 'configs/qwen3.5-2b.yaml'),
    ('qwen3.5-2b', 'pruned', 'configs/qwen3.5-2b-pruned.yaml'),
    ('granite-4.2-3b', 'enhanced', 'configs/granite-4.2-3b.yaml'),
]

SPLIT = 'test'
REFERENCE_LIMIT = 500         # Reference EVAL_LIMIT; the rows are re-derived and checked against the Hub jobs
EDGE_LIMIT = 100              # Rows scored on the CPU: a seeded subset of the reference rows
EDGE_SUBSET = 'random'        # 'random' or 'stratified' (keeps the reference share of rows with a database)
EDGE_SUBSET_SEED = 20261015

# Export: merge -> f16 GGUF -> quantise (only for runs with unfinished work in this session)
MERGE_DTYPE = 'float32'       # Exact LoRA merge before conversion; the GGUF intermediate is f16
KEEP_MERGED = False           # Delete merged weights after conversion (disk)
KEEP_F16 = False              # Delete the f16 GGUF after quantising unless F16 is benchmarked
UPLOAD_GGUF = False           # Upload GGUFs (UPLOAD_GGUF_QUANTS) to edge-eval/<edge_tag>/gguf/
UPLOAD_GGUF_QUANTS = ['Q4_K_M']
EXPORT_MINUTES_ESTIMATE = 30  # Per run (merge + convert + quantise); no export starts with less time left

# Accuracy jobs on the CPU (queued by priority; later ones run only if time remains)
ACCURACY_QUANT = 'Q4_K_M'     # Priority 0: every run. Priority 1: cross-schema (enhanced-trained, exact-match prompts)
RUN_CROSS_SCHEMA = True
EXTRA_ACCURACY_QUANTS = ['Q8_0']      # Priority 2: quantisation fidelity
EXTRA_QUANT_VARIANTS = ('enhanced',)  # ...for these schema variants only
EDGE_STOP_MODE = 'hf'         # 'hf': stop on EOS only, as the GPU jobs; 'config': also generation.stop_strings
DEFAULT_S_PER_ITEM = 30       # Planning estimate before anything has been measured

# Throughput and footprint (one run per model: speed does not depend on the adapter)
LATENCY_QUANTS = ['Q4_K_M', 'Q5_K_M', 'Q8_0', 'F16']
THREAD_SWEEP = [1, 2, 4]      # Capped at the CPUs available; the full count is always included
THREAD_SWEEP_QUANTS = ['Q4_K_M']      # Other quantisations are measured at the full thread count only
BENCH_PROMPT_TOKENS = 512     # llama-bench ppN
BENCH_GEN_TOKENS = 64         # llama-bench tgN
BENCH_REPS = 3
BENCH_TIMEOUT_S = 3600
TYPICAL_GEN_TOKENS = 48       # With the median edge prompt length, picks the fastest thread count for 'auto'
SERVER_THREADS = 'auto'       # 'auto' or an integer
CACHE_PROMPT = False          # True lets llama-server reuse the shared instruction prefix between queries
REQUEST_TIMEOUT_S = 900

# Scoring and statistics
RUN_EXECUTION = True
RUN_SYNTAX_CHECK = True
DEAD_DB_POLICY = 'flag'
BOOTSTRAP_SAMPLES = 2000
CI_LEVEL = 0.95
STATS_SEED = 20261015
EQUIV_MARGIN = 0.05           # TOST margin on CPU vs GPU execution-accuracy differences (absolute)

RUN_SUPPLEMENTARY = True      # Analyses of the GPU reference jobs that final-eval does not produce

SESSION_BUDGET_MINUTES = 720  # Kaggle session cap (12 h)
RESERVE_MINUTES = 45          # Kept for supplementary analysis, aggregation, figures and uploads

UPLOAD_TO_HUB = True
NTFY_TOPIC = 'cxmrykk'        # Blank disables push notifications
NTFY_SERVER = 'https://ntfy.sh'

SCRATCH_DIR = '/kaggle/temp'    # Repo, models, GGUFs (not saved; falls back to /tmp)
OUTPUT_DIR = '/kaggle/working'  # Logs and the results bundle (saved with the notebook version)

if EDGE_SUBSET not in ('random', 'stratified'):
    raise ValueError("EDGE_SUBSET must be 'random' or 'stratified'.")
if EDGE_STOP_MODE not in ('hf', 'config'):
    raise ValueError("EDGE_STOP_MODE must be 'hf' or 'config'.")
if EDGE_LIMIT > REFERENCE_LIMIT:
    raise ValueError('EDGE_LIMIT cannot exceed REFERENCE_LIMIT.')
print('Reference tag : ' + REFERENCE_EVAL_TAG)
print('Edge rows     : ' + str(EDGE_LIMIT) + ' of ' + str(REFERENCE_LIMIT) + ' (' + EDGE_SUBSET + ')')
print('Accuracy      : ' + ACCURACY_QUANT + ' (all runs)' + (', cross-schema' if RUN_CROSS_SCHEMA else '')
      + ', extra ' + str(EXTRA_ACCURACY_QUANTS) + ' for ' + str(EXTRA_QUANT_VARIANTS))
print('Latency       : ' + str(LATENCY_QUANTS) + ', threads ' + str(THREAD_SWEEP) + ' for ' + str(THREAD_SWEEP_QUANTS))
print('Budget        : ' + str(SESSION_BUDGET_MINUTES) + ' min (reserve ' + str(RESERVE_MINUTES) + ' min)')

In [ ]:
# 2. Platform checks (Kaggle CPU session), helpers, notifications and Hub access
import atexit
import codecs
import datetime
import hashlib
import json
import os
import platform
import shutil
import signal
import subprocess
import sys
import urllib.request
from pathlib import Path

CR, LF = chr(13), chr(10)
PLATFORM = 'kaggle'

ON_KAGGLE = Path('/kaggle').is_dir() or bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
if not ON_KAGGLE:
    raise RuntimeError('This notebook targets a Kaggle CPU session.')
RUN_TYPE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'unknown')  # Interactive | Batch (commit)


def _gpu_names():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=20).stdout
    except Exception:
        return []
    return [ln.strip() for ln in out.splitlines() if ln.strip()]


GPUS_PRESENT = _gpu_names()
if GPUS_PRESENT:
    print('WARNING: a GPU accelerator is attached (' + ', '.join(GPUS_PRESENT) + '). This notebook never uses it; '
          'select Settings > Accelerator > None to keep GPU quota.')
# Every child process (merge, conversion, tokenisation) runs on the CPU only.
os.environ['CUDA_VISIBLE_DEVICES'] = ''


def _lscpu():
    try:
        out = subprocess.run(['lscpu'], capture_output=True, text=True, timeout=20).stdout
    except Exception:
        return {}
    info = {}
    for line in out.splitlines():
        if ':' in line:
            key, value = line.split(':', 1)
            info[key.strip()] = value.strip()
    return info


def _cgroup_cpu_limit():
    try:
        quota, period = Path('/sys/fs/cgroup/cpu.max').read_text().split()[:2]
        return None if quota == 'max' else round(int(quota) / int(period), 2)
    except Exception:
        pass
    try:
        quota = int(Path('/sys/fs/cgroup/cpu/cpu.cfs_quota_us').read_text())
        period = int(Path('/sys/fs/cgroup/cpu/cpu.cfs_period_us').read_text())
        return None if quota <= 0 else round(quota / period, 2)
    except Exception:
        return None


def _meminfo_gb(key):
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            if line.startswith(key + ':'):
                return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return float('nan')


LSCPU = _lscpu()
CPU_MODEL = LSCPU.get('Model name') or platform.processor() or 'unknown'
try:
    LOGICAL_CPUS = len(os.sched_getaffinity(0))
except Exception:
    LOGICAL_CPUS = os.cpu_count() or 1
try:
    PHYSICAL_CORES = int(LSCPU.get('Core(s) per socket', '0')) * int(LSCPU.get('Socket(s)', '1')) or None
except ValueError:
    PHYSICAL_CORES = None
CPU_FLAGS = set(LSCPU.get('Flags', '').split())
SIMD_FLAGS = [f for f in ('sse4_2', 'avx', 'avx2', 'fma', 'f16c', 'avx512f', 'avx512_vnni', 'avx512_bf16',
                          'avx_vnni', 'amx_tile') if f in CPU_FLAGS]
CGROUP_CPU_LIMIT = _cgroup_cpu_limit()
RAM_TOTAL_GB = _meminfo_gb('MemTotal')


def _internet_ok(url='https://huggingface.co', timeout=15):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            return resp.status < 500
    except Exception:
        return False


if not _internet_ok():
    raise RuntimeError('No internet access. Settings > Internet > On (phone-verified account).')


def _writable_dir(path):
    try:
        p = Path(path)
        p.mkdir(parents=True, exist_ok=True)
        probe = p / '.t2c-write-test'
        probe.write_text('ok', encoding='utf-8')
        probe.unlink()
        return True
    except Exception:
        return False


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float('nan')


WORK_ROOT = Path(SCRATCH_DIR) if _writable_dir(SCRATCH_DIR) else Path('/tmp')
REPO_DIR = WORK_ROOT / 'capstone'
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S') + '-kaggle-cpu'
LOG_DIR = Path(OUTPUT_DIR) / 'edge-eval-logs' / SESSION
LOG_DIR.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Could not read the HF_TOKEN Kaggle secret (' + type(exc).__name__ + '). '
                       'Add-ons > Secrets: add HF_TOKEN and attach it.') from exc

os.environ['T2C_HUB_REPO'] = HUB_REPO
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

STATUS = {'session': SESSION, 'platform': PLATFORM, 'run_type': RUN_TYPE, 'cpu': CPU_MODEL,
          'current_step': None, 'jobs': {}, 'outcome': None}

# --------------------------------------------------------------------------- #
# ntfy push notifications (best-effort; the topic is public, so no secrets)
# --------------------------------------------------------------------------- #
NTFY_URL = (NTFY_SERVER.rstrip('/') + '/' + NTFY_TOPIC.strip()) if NTFY_TOPIC.strip() else None


def notify(title, message, priority='default', tags=()):
    if not NTFY_URL:
        return False
    headers = {'Title': ('T2C edge eval: ' + title).encode('ascii', 'replace').decode('ascii'),
               'Priority': str(priority)}
    if tags:
        headers['Tags'] = ','.join(tags)
    body = message if len(message) <= 3000 else '...' + message[-3000:]
    req = urllib.request.Request(NTFY_URL, data=body.encode('utf-8'), headers=headers, method='POST')
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            resp.read()
        return True
    except Exception as exc:
        print('[ntfy] Notification failed: ' + str(exc), file=sys.stderr)
        return False


# --------------------------------------------------------------------------- #
# Step runner: streams to the cell and to a log file (tqdm redraws collapsed)
# --------------------------------------------------------------------------- #
def _clean_lines(buf, text):
    text = buf + text.replace(CR + LF, LF)
    parts = text.split(LF)
    return [p.split(CR)[-1] for p in parts[:-1]], parts[-1].split(CR)[-1]


def run_step(name, cmd, log_path=None, allow_fail=False, env=None, cwd=None, echo=True):
    log_path = Path(log_path) if log_path else LOG_DIR / (name + '.log')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    workdir = cwd or (str(REPO_DIR) if REPO_DIR.exists() else str(WORK_ROOT))
    STATUS['current_step'] = name
    print('=' * 72)
    print('[' + name + '] ' + cmd)
    print('    log: ' + str(log_path))
    print('=' * 72, flush=True)
    t0 = time.time()
    buf = ''
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    with open(log_path, 'w', encoding='utf-8') as fh:
        fh.write('$ ' + cmd + LF + '# cwd: ' + workdir + LF + LF)
        fh.flush()
        proc = subprocess.Popen(cmd, shell=True, cwd=workdir, env={**os.environ, **(env or {})},
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            while True:
                chunk = proc.stdout.read1(8192)
                if not chunk:
                    break
                text = decoder.decode(chunk)
                if echo:
                    sys.stdout.write(text)
                    sys.stdout.flush()
                lines, buf = _clean_lines(buf, text)
                for line in lines:
                    fh.write(line + LF)
                fh.flush()
            rc = proc.wait()
        except KeyboardInterrupt:
            try:
                os.killpg(proc.pid, signal.SIGINT)
                proc.wait(timeout=60)
            except Exception:
                proc.kill()
            raise
        if buf:
            fh.write(buf + LF)
        elapsed = time.time() - t0
        fh.write(LF + '# exit code: ' + str(rc) + LF + '# elapsed: ' + format(elapsed, '.1f') + 's' + LF)
    STATUS['current_step'] = None
    print('[' + name + '] exit ' + str(rc) + ' after ' + format(elapsed / 60, '.1f') + ' min', flush=True)
    if rc != 0 and not allow_fail:
        raise RuntimeError('Step ' + name + ' failed with exit code ' + str(rc) + '. See ' + str(log_path))
    return rc


def minutes_left():
    """Minutes left before the reserve kept for analysis and uploads."""
    return SESSION_BUDGET_MINUTES - RESERVE_MINUTES - (time.time() - PIPELINE_T0) / 60


def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        while True:
            block = fh.read(chunk)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


# --------------------------------------------------------------------------- #
# Hub access
# --------------------------------------------------------------------------- #
_HUB = {'files': None}


def hub_api():
    from huggingface_hub import HfApi
    return HfApi(token=os.environ['HF_TOKEN'])


def hub_files(refresh=False):
    if _HUB['files'] is None or refresh:
        _HUB['files'] = set(hub_api().list_repo_files(HUB_REPO, repo_type='model'))
    return _HUB['files']


def hub_download_file(path_in_repo):
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(repo_id=HUB_REPO, filename=path_in_repo, repo_type='model',
                                token=os.environ['HF_TOKEN']))


def hub_download_patterns(patterns, dest):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=HUB_REPO, repo_type='model', token=os.environ['HF_TOKEN'],
                      allow_patterns=list(patterns), local_dir=str(dest))
    return Path(dest)


def hub_upload_folder(local_dir, path_in_repo, message):
    """Uploads a folder in ONE commit, with retries. Never raises."""
    if not UPLOAD_TO_HUB:
        return False
    for attempt in range(3):
        try:
            hub_api().upload_folder(folder_path=str(local_dir), path_in_repo=path_in_repo,
                                    repo_id=HUB_REPO, repo_type='model', commit_message=message)
            return True
        except Exception as exc:
            print('[hub] Upload of ' + path_in_repo + ' failed (attempt ' + str(attempt + 1) + '/3): ' + str(exc),
                  flush=True)
            time.sleep(30 * (attempt + 1))
    return False


def hub_upload_file(local_path, path_in_repo, message):
    """Uploads one file in one commit, with retries. Never raises."""
    if not UPLOAD_TO_HUB:
        return False
    for attempt in range(3):
        try:
            hub_api().upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                                  repo_id=HUB_REPO, repo_type='model', commit_message=message)
            return True
        except Exception as exc:
            print('[hub] Upload of ' + path_in_repo + ' failed (attempt ' + str(attempt + 1) + '/3): ' + str(exc),
                  flush=True)
            time.sleep(30 * (attempt + 1))
    return False


# --------------------------------------------------------------------------- #
# Crash hooks
# --------------------------------------------------------------------------- #
def _notify_unfinished(reason):
    if STATUS.get('outcome') is not None:
        return
    STATUS['outcome'] = 'runtime-ended'
    done = [k for k, v in STATUS['jobs'].items() if v == 'completed']
    notify('runtime ended', reason + ' before the final cell (step: ' + str(STATUS.get('current_step')) + '). Session '
           + SESSION + '. Completed jobs: ' + (', '.join(done) or 'none')
           + '. Commit a new version: finished work is reused from the Hub.', priority='urgent', tags=['skull'])


def _on_sigterm(signum, frame):
    _notify_unfinished('The Kaggle session was shut down (SIGTERM)')
    prev = globals().get('_PREV_SIGTERM')
    if callable(prev):
        prev(signum, frame)
    else:
        signal.signal(signal.SIGTERM, signal.SIG_DFL)
        os.kill(os.getpid(), signal.SIGTERM)


def _on_cell_exception(shell, etype, evalue, tb, tb_offset=None):
    notify('notebook error', etype.__name__ + ': ' + str(evalue) + LF + 'Session ' + SESSION,
           priority='urgent', tags=['rotating_light'])
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


if not globals().get('_HOOKS_INSTALLED'):
    _PREV_SIGTERM = signal.getsignal(signal.SIGTERM)
    signal.signal(signal.SIGTERM, _on_sigterm)
    atexit.register(lambda: _notify_unfinished('The notebook kernel exited'))
    _HOOKS_INSTALLED = True
get_ipython().set_custom_exc((Exception,), _on_cell_exception)

print('Platform   : Kaggle (' + RUN_TYPE + ' session, CPU only)')
print('CPU        : ' + CPU_MODEL)
print('CPUs       : ' + str(LOGICAL_CPUS) + ' logical, ' + str(PHYSICAL_CORES) + ' physical, cgroup limit '
      + str(CGROUP_CPU_LIMIT))
print('SIMD       : ' + ', '.join(SIMD_FLAGS))
print('RAM        : ' + format(RAM_TOTAL_GB, '.1f') + ' GB')
print('Work root  : ' + str(WORK_ROOT) + ' (' + format(_free_gb(WORK_ROOT), '.0f') + ' GB free, not saved)')
print('Output dir : ' + OUTPUT_DIR + ' (' + format(_free_gb(OUTPUT_DIR), '.0f') + ' GB free, saved)')
print('Session    : ' + SESSION)
notify('started', 'Session ' + SESSION + ' (' + RUN_TYPE + ') on ' + CPU_MODEL + ', ' + str(LOGICAL_CPUS) + ' CPUs.',
       priority='low', tags=['rocket'])

In [ ]:
# 3. Clone the repository and pin the code version
if (REPO_DIR / '.git').exists():
    run_step('git-fetch', 'git -C ' + str(REPO_DIR) + ' fetch --all --tags --prune', cwd=str(WORK_ROOT),
             allow_fail=True)
    if not GIT_REF:
        run_step('git-pull', 'git -C ' + str(REPO_DIR) + ' pull --ff-only', cwd=str(WORK_ROOT), allow_fail=True)
else:
    run_step('git-clone', 'git clone ' + GITHUB_REPO + ' ' + str(REPO_DIR), cwd=str(WORK_ROOT))
if GIT_REF:
    run_step('git-checkout', 'git -C ' + str(REPO_DIR) + ' checkout ' + GIT_REF, cwd=str(WORK_ROOT))

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def _git(*args):
    return subprocess.run(['git', *args], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()


GIT_COMMIT = _git('rev-parse', 'HEAD')
GIT_DIRTY = bool(_git('status', '--porcelain'))
if not GIT_REF:
    print('WARNING: GIT_REF is blank, so this bundle is not pinned to a tag. Set GIT_REF for thesis runs.')

# The merge must use the verified adapter loader (history/027.md) and prompts must
# follow the training budget (history/030.md), exactly as in the GPU reference jobs.
loader_src = (REPO_DIR / 'src' / 'training' / 'model_loader.py').read_text(encoding='utf-8')
dataset_src = (REPO_DIR / 'src' / 'data' / 'dataset.py').read_text(encoding='utf-8')
neo4j_src = (REPO_DIR / 'src' / 'evaluation' / 'neo4j_client.py').read_text(encoding='utf-8')
if 'def attach_adapter' not in loader_src or 'Adapter verified' not in loader_src:
    raise RuntimeError('This checkout lacks the verified adapter loader (history/027.md).')
if 'def fit_prompt_to_budget' not in dataset_src or 'connection_failed' not in neo4j_src:
    raise RuntimeError('This checkout lacks the history/030.md changes (prompt budget, Neo4j fixes).')
print('Commit : ' + GIT_COMMIT + (' (uncommitted changes)' if GIT_DIRTY else ''))

In [ ]:
# 4. Dependencies, dataset and base models, and a CPU build of llama.cpp
# Unsloth is not needed: merging loads adapters through Transformers + PEFT.
run_step('bootstrap', 'bash scripts/colab_bootstrap.sh', env={'INSTALL_UNSLOTH': '0'})

# PEFT must be able to inject LoRA layers on this image (history/033.md). An old
# preinstalled torchao makes PEFT raise ImportError during adapter loading. The
# bootstrap removes it; this check fails in seconds instead of after loading
# every base model in the export cell.
run_step('peft-check', 'python -c "from peft.import_utils import is_torchao_available; '
                       'print(\'torchao usable by PEFT:\', is_torchao_available())"')

DOWNLOAD_TARGETS = {'gemma-4-e2b': 'gemma', 'qwen3.5-2b': 'qwen', 'granite-4.2-3b': 'granite'}
for target in dict.fromkeys(DOWNLOAD_TARGETS[m] for m, _v, _c in EXPERIMENTS):
    run_step('download-' + target, 'bash download_data.sh ' + target)

# --------------------------------------------------------------------------- #
# llama.cpp: pinned commit (LLAMA_CPP_REF, else the commit the first session recorded)
# --------------------------------------------------------------------------- #
LLAMA_DIR = WORK_ROOT / 'llama.cpp'
LLAMA_BIN = LLAMA_DIR / 'build' / 'bin'
LLAMA_PIN_PATH = 'edge-eval/' + EDGE_TAG_SUFFIX + '/llama_cpp_commit.txt'

pinned = None
if not LLAMA_CPP_REF:
    try:
        if LLAMA_PIN_PATH in hub_files(refresh=True):
            pinned = hub_download_file(LLAMA_PIN_PATH).read_text(encoding='utf-8').strip() or None
            print('Using the llama.cpp commit pinned by an earlier session: ' + str(pinned))
    except Exception as exc:
        print('Could not read the llama.cpp pin (' + str(exc) + '); building the latest commit.')
LLAMA_REF_USED = LLAMA_CPP_REF or pinned

run_step('build-tools', 'command -v cmake >/dev/null && command -v g++ >/dev/null || '
                        '(apt-get update -qq && apt-get install -y -qq cmake build-essential)', allow_fail=True)
if not (LLAMA_DIR / '.git').exists():
    if LLAMA_REF_USED:
        run_step('llama-clone', 'git clone --filter=blob:none ' + LLAMA_CPP_REPO + ' ' + str(LLAMA_DIR),
                 cwd=str(WORK_ROOT))
    else:
        run_step('llama-clone', 'git clone --depth 1 ' + LLAMA_CPP_REPO + ' ' + str(LLAMA_DIR), cwd=str(WORK_ROOT))
if LLAMA_REF_USED:
    run_step('llama-checkout', 'git -C ' + str(LLAMA_DIR) + ' checkout --detach ' + LLAMA_REF_USED, cwd=str(WORK_ROOT))

# GGML_NATIVE tunes kernels to this CPU, as an edge build would be tuned to its device.
run_step('llama-cmake', 'cmake -B ' + str(LLAMA_DIR / 'build') + ' -S ' + str(LLAMA_DIR)
         + ' -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=ON -DGGML_CUDA=OFF -DLLAMA_CURL=OFF', cwd=str(WORK_ROOT))
run_step('llama-build', 'cmake --build ' + str(LLAMA_DIR / 'build') + ' --config Release -j ' + str(LOGICAL_CPUS)
         + ' --target llama-server llama-quantize llama-bench', cwd=str(WORK_ROOT))
for binary in ('llama-server', 'llama-quantize', 'llama-bench'):
    if not (LLAMA_BIN / binary).exists():
        raise RuntimeError(binary + ' was not built; see the llama-build log.')

# The converter uses the checkout's own gguf-py. Its full requirements file pins older
# transformers releases, so only the light dependencies are installed and the floor re-asserted.
run_step('llama-convert-deps', 'python -m pip install --quiet ' + str(LLAMA_DIR / 'gguf-py')
         + ' sentencepiece protobuf && python -m pip install --quiet "transformers>=5.5.0"')
os.environ['LLAMA_CPP_DIR'] = str(LLAMA_DIR)

LLAMA_COMMIT = subprocess.run(['git', '-C', str(LLAMA_DIR), 'rev-parse', 'HEAD'],
                              capture_output=True, text=True).stdout.strip()
_ver = subprocess.run([str(LLAMA_BIN / 'llama-server'), '--version'], capture_output=True, text=True, timeout=60)
LLAMA_VERSION = ' | '.join(ln.strip() for ln in (_ver.stdout + _ver.stderr).splitlines() if ln.strip())[:300]
print('llama.cpp commit : ' + LLAMA_COMMIT)
print('llama.cpp version: ' + LLAMA_VERSION)

if not LLAMA_CPP_REF and not pinned:
    pin_file = LOG_DIR / 'llama_cpp_commit.txt'
    pin_file.write_text(LLAMA_COMMIT + LF, encoding='utf-8')
    if hub_upload_file(pin_file, LLAMA_PIN_PATH, '[edge-eval] pin llama.cpp ' + LLAMA_COMMIT[:10]):
        print('Recorded this llama.cpp commit on the Hub; later sessions will build the same one.')

run_step('doctor', 'python app.py doctor', allow_fail=True)
print('Free disk in work root: ' + format(_free_gb(WORK_ROOT), '.0f') + ' GB')

In [ ]:
# 5. Check the Neo4j Labs demo server (execution accuracy depends on it)
if RUN_EXECUTION:
    rc = run_step('neo4j-check', 'python app.py neo4j check --alias neo4jlabs_demo_db_movies', allow_fail=True)
    if rc != 0:
        raise RuntimeError('The Neo4j Labs demo server is unreachable from this Kaggle session (outbound Bolt on '
                           'port 7687). Retry later, or set RUN_EXECUTION = False to report text metrics only.')
else:
    print('RUN_EXECUTION is False: execution accuracy will not be measured.')

In [ ]:
# 6. Runs and adapters, the GPU reference jobs, and the edge row subset
import random

import pandas as pd
import yaml
from src.config import load_config
from src.data.dataset import load_raw_split, subsample

EXP = []
for model, variant, config in EXPERIMENTS:
    raw = yaml.safe_load(Path(config).read_text(encoding='utf-8'))
    cfg = load_config(config)
    EXP.append({'model': model, 'variant': variant, 'config': config, 'run_name': raw['run_name'],
                'schema_mode': cfg.data.schema_mode, 'seed': cfg.data.seed,
                'fingerprint_now': cfg.fingerprint(),
                'adapter_dir': REPO_DIR / 'artifacts' / 'runs' / raw['run_name'] / 'final'})
seeds = sorted({e['seed'] for e in EXP})
if len(seeds) != 1:
    raise RuntimeError('Configs use different data seeds ' + str(seeds) + '; rows would not match the reference.')
SAMPLE_SEED = seeds[0]

for e in EXP:
    adir = e['adapter_dir']
    if not (adir / 'adapter_config.json').exists():
        rc = run_step('pull-' + e['run_name'], 'python app.py checkpoint pull --config ' + e['config'] + ' --step final',
                      allow_fail=True)
        if rc not in (0, 1):
            raise RuntimeError(e['run_name'] + ': fetching the final adapter failed with exit ' + str(rc)
                               + ' (Hub error, not "not found"). Rerun when the Hub is reachable.')
    e['adapter_ok'] = (adir / 'adapter_config.json').exists() and any(adir.glob('adapter_model.*'))
    meta_path = adir / 'run_metadata.json'
    e['train_meta'] = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    fp = e['train_meta'].get('fingerprint')
    e['fingerprint_trained'] = fp
    e['fingerprint_match'] = (fp == e['fingerprint_now']) if fp else None
    print(e['run_name'].ljust(30) + ('adapter OK' if e['adapter_ok'] else 'adapter MISSING').ljust(18)
          + 'fingerprint ' + {True: 'match', False: 'DIFFERS', None: 'n/a'}[e['fingerprint_match']])

# Schema-filter parameters used by the exact-match runs (for the supplementary analyses).
_pruned_cfg = next((load_config(c) for _m, v, c in EXPERIMENTS if v == 'pruned'), load_config(EXPERIMENTS[0][2]))
FILTER_MODE = _pruned_cfg.data.schema_mode
FILTER_KW = dict(min_nodes=_pruned_cfg.data.filter_min_nodes,
                 keep_patterns_for_kept_nodes=_pruned_cfg.data.filter_keep_all_patterns_for_kept_nodes,
                 similarity_threshold=_pruned_cfg.data.similarity_threshold,
                 similarity_model=_pruned_cfg.data.similarity_model)

# --------------------------------------------------------------------------- #
# GPU reference jobs (final-eval-kaggle)
# --------------------------------------------------------------------------- #
REF_PREFIX = 'final-eval/' + REFERENCE_EVAL_TAG
HUB_CACHE = REPO_DIR / 'artifacts' / 'hub_cache'
files = hub_files(refresh=True)
ref_ids = sorted({f.split('/')[3] for f in files
                  if f.startswith(REF_PREFIX + '/jobs/') and f.endswith('/job.json') and f.count('/') == 4})
if not ref_ids:
    raise RuntimeError('No reference jobs under ' + REF_PREFIX + '/jobs/ on the Hub. Run final-eval-kaggle first.')
hub_download_patterns([REF_PREFIX + '/jobs/*/job.json', REF_PREFIX + '/jobs/*/predictions.scored.jsonl'], HUB_CACHE)

REF = {}
for jid in ref_ids:
    d = HUB_CACHE / REF_PREFIX / 'jobs' / jid
    try:
        rec = json.loads((d / 'job.json').read_text(encoding='utf-8'))
    except Exception as exc:
        print('Skipping reference ' + jid + ': ' + str(exc))
        continue
    scored = d / 'predictions.scored.jsonl'
    if rec.get('status') != 'completed' or not scored.exists():
        continue
    rows = [json.loads(ln) for ln in scored.read_text(encoding='utf-8').splitlines() if ln.strip()]
    df = pd.DataFrame(rows)
    df['instance_id'] = df['instance_id'].astype(str)
    REF[jid] = {'job': rec, 'rows': df}
REF_COMMITS = sorted({str(v['job'].get('git_commit')) for v in REF.values()})
print()
print('Reference jobs loaded: ' + str(len(REF)) + ' (commits ' + ', '.join(c[:10] for c in REF_COMMITS) + ')')
if GIT_COMMIT not in REF_COMMITS:
    print('NOTE: the reference jobs ran on a different code commit. Prompt construction must be identical for the '
          'paired GPU/CPU comparison; prefer GIT_REF = the reference commit.')
for e in EXP:
    if e['run_name'] + '__fine_tuned' not in REF:
        print('WARNING: no GPU reference job ' + e['run_name'] + '__fine_tuned; its CPU job will not be paired.')

# --------------------------------------------------------------------------- #
# Rows: re-derive the reference sample, then a seeded subset for the CPU
# --------------------------------------------------------------------------- #
TEST_RECORDS = load_raw_split(SPLIT)
REF_RECORDS = subsample(TEST_RECORDS, REFERENCE_LIMIT, SAMPLE_SEED)
REF_IDS = [str(r.get('instance_id')) for r in REF_RECORDS]
REF_BY_ID = {str(r.get('instance_id')): r for r in REF_RECORDS}
REF_ROWS_MISMATCH = [jid for jid, v in REF.items()
                     if v['job'].get('backend', 'hf') == 'hf' and set(v['rows']['instance_id']) != set(REF_IDS)]
if REF_ROWS_MISMATCH:
    print('WARNING: re-derived rows differ from reference jobs: ' + ', '.join(REF_ROWS_MISMATCH))

rng = random.Random(EDGE_SUBSET_SEED)
k = min(EDGE_LIMIT, len(REF_RECORDS))
if EDGE_SUBSET == 'stratified':
    with_db = [i for i, r in enumerate(REF_RECORDS) if r.get('database_reference')]
    without_db = [i for i, r in enumerate(REF_RECORDS) if not r.get('database_reference')]
    k_db = min(len(with_db), round(k * len(with_db) / max(1, len(REF_RECORDS))))
    idx = rng.sample(with_db, k_db) + rng.sample(without_db, min(len(without_db), k - k_db))
else:
    idx = rng.sample(range(len(REF_RECORDS)), k)
EDGE_RECORDS = [REF_RECORDS[i] for i in sorted(idx)]
EDGE_IDS = [str(r.get('instance_id')) for r in EDGE_RECORDS]
EDGE_IDS_SHA = hashlib.sha256(LF.join(EDGE_IDS).encode('utf-8')).hexdigest()[:16]

EDGE_TAG = SPLIT + '-n' + str(len(EDGE_IDS)) + 'of' + str(REFERENCE_LIMIT) + '-' + EDGE_SUBSET + '-' + EDGE_TAG_SUFFIX
EDGE_PREFIX = 'edge-eval/' + EDGE_TAG
EDGE_DIR = REPO_DIR / 'artifacts' / 'edge-eval' / EDGE_TAG
EDGE_DIR.mkdir(parents=True, exist_ok=True)

n_db_ref = sum(1 for r in REF_RECORDS if r.get('database_reference'))
n_db_edge = sum(1 for r in EDGE_RECORDS if r.get('database_reference'))
print('Edge tag     : ' + EDGE_TAG + ' (rows sha ' + EDGE_IDS_SHA + ')')
print('Edge rows    : ' + str(len(EDGE_IDS)) + ' (' + str(n_db_edge) + ' with a database; reference '
      + str(n_db_ref) + '/' + str(len(REF_RECORDS)) + ')')

In [ ]:
# 7. Plan jobs, benchmarks and exports; reuse finished work from the Hub
QUANT_ORDER = ['F16', 'Q8_0', 'Q6_K', 'Q5_K_M', 'Q5_K_S', 'Q4_K_M', 'Q4_K_S', 'Q4_0', 'Q3_K_M', 'Q2_K']


def _qkey(q):
    return QUANT_ORDER.index(q) if q in QUANT_ORDER else len(QUANT_ORDER)


BY_VARIANT = {(e['model'], e['variant']): e for e in EXP}


def _job(e, condition, quant, prompt_mode, priority):
    ref_id = e['run_name'] + '__fine_tuned' + ('' if condition == 'fine_tuned' else '__' + prompt_mode + '_prompts')
    return {**e, 'condition': condition, 'quant': quant, 'schema_mode': prompt_mode,
            'prompt_schema_mode': prompt_mode, 'train_schema_mode': e['schema_mode'],
            'reference_job_id': ref_id, 'job_id': ref_id + '__gguf_' + quant.lower() + '_cpu', 'priority': priority}


JOBS = [_job(e, 'fine_tuned', ACCURACY_QUANT, e['schema_mode'], 0) for e in EXP]
if RUN_CROSS_SCHEMA:
    for (m, v), e in BY_VARIANT.items():
        pruned = BY_VARIANT.get((m, 'pruned'))
        if v == 'enhanced' and pruned is not None:
            JOBS.append(_job(e, 'cross_schema', ACCURACY_QUANT, pruned['schema_mode'], 1))
for q in EXTRA_ACCURACY_QUANTS:
    if q == ACCURACY_QUANT:
        continue
    for e in EXP:
        if e['variant'] in EXTRA_QUANT_VARIANTS:
            JOBS.append(_job(e, 'fine_tuned', q, e['schema_mode'], 2))
JOBS.sort(key=lambda j: j['priority'])

# One run per model for throughput/footprint: speed depends on the architecture, not the adapter.
BENCH_PLAN = {}
for e in sorted(EXP, key=lambda x: (not x['adapter_ok'], x['variant'] != 'enhanced')):
    BENCH_PLAN.setdefault(e['model'], e)


def _provenance():
    return {'edge_tag': EDGE_TAG, 'edge_ids_sha': EDGE_IDS_SHA, 'n_planned': len(EDGE_IDS),
            'reference_eval_tag': REFERENCE_EVAL_TAG, 'git_ref': GIT_REF, 'git_commit': GIT_COMMIT,
            'git_dirty': GIT_DIRTY, 'llama_cpp_commit': LLAMA_COMMIT, 'llama_cpp_version': LLAMA_VERSION,
            'cpu_model': CPU_MODEL, 'logical_cpus': LOGICAL_CPUS, 'physical_cores': PHYSICAL_CORES,
            'simd_flags': SIMD_FLAGS, 'cgroup_cpu_limit': CGROUP_CPU_LIMIT,
            'ram_total_gb': round(RAM_TOTAL_GB, 1), 'platform': PLATFORM, 'session': SESSION}


def _reuse_mismatch(rec):
    checks = [('edge_ids_sha', EDGE_IDS_SHA), ('llama_cpp_commit', LLAMA_COMMIT), ('cpu_model', CPU_MODEL)]
    if REUSE_SAME_COMMIT_ONLY:
        checks.append(('git_commit', GIT_COMMIT))
    for key, want in checks:
        if rec.get(key) != want:
            return key + ' differs (' + str(rec.get(key))[:24] + ' vs ' + str(want)[:24] + ')'
    return None


def _job_record(job, **extra):
    keep = ('job_id', 'reference_job_id', 'model', 'variant', 'config', 'run_name', 'schema_mode', 'condition',
            'quant', 'prompt_schema_mode', 'train_schema_mode', 'priority', 'fingerprint_now',
            'fingerprint_trained', 'fingerprint_match')
    rec = {k: job.get(k) for k in keep}
    rec.update(_provenance())
    rec['backend'] = 'llamacpp'
    rec.update(extra)
    return rec


JOB_RESULTS = {}
BENCH_RESULTS = {}
EXPORT_RECORDS = {}
SKIPPED_FOR_TIME = []
files = hub_files(refresh=True)

# Export metadata from earlier sessions (sizes, hashes, merge verification).
for path in sorted(f for f in files if f.startswith(EDGE_PREFIX + '/exports/') and f.endswith('.json')):
    try:
        rec = json.loads(hub_download_file(path).read_text(encoding='utf-8'))
        rec['from_hub'] = True
        EXPORT_RECORDS[rec['run_name']] = rec
    except Exception as exc:
        print('Could not read ' + path + ': ' + str(exc))

PENDING_JOBS = []
for job in JOBS:
    jid = job['job_id']
    if not job['adapter_ok']:
        JOB_RESULTS[jid] = _job_record(job, status='missing_adapter', error='no final adapter on the Hub')
        print('[' + jid + '] cannot run: no final adapter.')
        continue
    prefix = EDGE_PREFIX + '/jobs/' + jid
    reused = None
    if SKIP_COMPLETED_JOBS and (prefix + '/job.json') in files:
        rec = json.loads(hub_download_file(prefix + '/job.json').read_text(encoding='utf-8'))
        why = _reuse_mismatch(rec) if rec.get('status') == 'completed' else 'status ' + str(rec.get('status'))
        if why:
            print('[' + jid + '] Hub outputs not reused: ' + why + '.')
        else:
            hub_download_patterns([prefix + '/*'], HUB_CACHE)
            jdir = EDGE_DIR / 'jobs' / jid
            if jdir.exists():
                shutil.rmtree(jdir)
            shutil.copytree(HUB_CACHE / prefix, jdir)
            if (jdir / 'predictions.scored.jsonl').exists():
                reused = rec
    if reused is not None:
        JOB_RESULTS[jid] = reused
        STATUS['jobs'][jid] = 'completed'
        print('[' + jid + '] reusing completed outputs from the Hub.')
    else:
        PENDING_JOBS.append(job)

PENDING_BENCH = []
for model, e in BENCH_PLAN.items():
    if not e['adapter_ok']:
        BENCH_RESULTS[model] = {'model': model, 'run_name': e['run_name'], 'status': 'missing_adapter'}
        continue
    path = EDGE_PREFIX + '/bench/' + model + '/bench.json'
    if SKIP_COMPLETED_JOBS and path in files:
        rec = json.loads(hub_download_file(path).read_text(encoding='utf-8'))
        why = _reuse_mismatch(rec)
        if not why:
            BENCH_RESULTS[model] = rec
            print('[bench ' + model + '] reusing results from the Hub.')
            continue
        print('[bench ' + model + '] Hub results not reused: ' + why + '.')
    PENDING_BENCH.append(model)

NEEDED_QUANTS = {}
for job in PENDING_JOBS:
    NEEDED_QUANTS.setdefault(job['run_name'], set()).add(job['quant'])
for model in PENDING_BENCH:
    NEEDED_QUANTS.setdefault(BENCH_PLAN[model]['run_name'], set()).update(LATENCY_QUANTS + [ACCURACY_QUANT])

print()
print('Jobs planned : ' + str(len(JOBS)) + ' | reused ' + str(sum(1 for r in JOB_RESULTS.values()
                                                                   if r.get('status') == 'completed'))
      + ' | pending ' + str(len(PENDING_JOBS)))
for job in PENDING_JOBS:
    print('  [p' + str(job['priority']) + '] ' + job['job_id'])
print('Benchmarks pending: ' + (', '.join(PENDING_BENCH) or 'none'))
for rn, qs in NEEDED_QUANTS.items():
    print('  export ' + rn + ': ' + ', '.join(sorted(qs, key=_qkey)))

In [ ]:
# 8. Export: merge (verified) -> f16 GGUF -> quantise, only for runs with unfinished work
GGUF_DIR = WORK_ROOT / 'gguf'
MERGED_ROOT = WORK_ROOT / 'merged'
QUANTIZE_BIN = LLAMA_BIN / 'llama-quantize'


def _dir_bytes(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())


def export_run(e, quants):
    rn = e['run_name']
    out_dir = GGUF_DIR / rn
    out_dir.mkdir(parents=True, exist_ok=True)
    merged = MERGED_ROOT / rn
    f16 = out_dir / (rn + '-f16.gguf')
    rec = {'run_name': rn, 'model': e['model'], 'variant': e['variant'], 'merge_dtype': MERGE_DTYPE,
           'requested_quants': sorted(quants, key=_qkey), 'status': 'started', 'quants': {}, **_provenance()}
    EXPORT_RECORDS[rn] = rec
    try:
        print('Free disk before export: ' + format(_free_gb(WORK_ROOT), '.0f') + ' GB')
        t0 = time.time()
        merge_log = LOG_DIR / ('merge-' + rn + '.log')
        run_step('merge-' + rn, 'python app.py export merge --config ' + e['config'] + ' --adapter '
                 + str(e['adapter_dir']) + ' --out ' + str(merged) + ' --dtype ' + MERGE_DTYPE, log_path=merge_log)
        rec['merge_s'] = round(time.time() - t0, 1)
        log_lines = merge_log.read_text(encoding='utf-8', errors='replace').splitlines()
        rec['adapter_verified'] = [ln.split('| ', 1)[-1] for ln in log_lines if 'Adapter verified:' in ln]
        rec['adapter_remapped'] = [ln.split('| ', 1)[-1] for ln in log_lines
                                   if 'Remapped' in ln and 'adapter tensor' in ln]
        if not rec['adapter_verified']:
            raise RuntimeError('The merge log has no Adapter verified line; refusing to export a possibly '
                               'unadapted model (history/027.md).')
        rec['merge_metadata'] = json.loads((merged / 'merge_metadata.json').read_text(encoding='utf-8'))
        rec['merged_size_bytes'] = _dir_bytes(merged)

        t0 = time.time()
        run_step('convert-' + rn, 'python app.py export gguf --merged ' + str(merged) + ' --out-dir '
                 + str(out_dir) + ' --quant F16', log_path=LOG_DIR / ('convert-' + rn + '.log'))
        rec['convert_s'] = round(time.time() - t0, 1)
        if not f16.exists():
            raise RuntimeError('Conversion finished without ' + f16.name + '.')
        rec['f16_size_bytes'] = f16.stat().st_size
        if not KEEP_MERGED:
            shutil.rmtree(merged, ignore_errors=True)

        for q in sorted(quants, key=_qkey):
            if q == 'F16':
                path, qs = f16, 0.0
            else:
                path = out_dir / (rn + '-' + q.lower() + '.gguf')
                t0 = time.time()
                run_step('quantize-' + rn + '-' + q.lower(),
                         str(QUANTIZE_BIN) + ' ' + str(f16) + ' ' + str(path) + ' ' + q + ' ' + str(LOGICAL_CPUS),
                         log_path=LOG_DIR / ('quantize-' + rn + '-' + q.lower() + '.log'), echo=False)
                qs = round(time.time() - t0, 1)
            rec['quants'][q] = {'path': str(path), 'file': path.name, 'size_bytes': path.stat().st_size,
                                'sha256': sha256_file(path), 'quantize_s': qs}
            print('  ' + q.ljust(8) + format(path.stat().st_size / 1e6, '9.0f') + ' MB  ' + path.name)
            if UPLOAD_GGUF and q in UPLOAD_GGUF_QUANTS:
                hub_upload_file(path, EDGE_PREFIX + '/gguf/' + rn + '/' + path.name, '[edge-eval] ' + path.name)
        if 'F16' not in quants and not KEEP_F16:
            f16.unlink(missing_ok=True)
        rec['status'] = 'ok'
    except KeyboardInterrupt:
        rec['status'] = 'interrupted'
        raise
    except Exception as exc:
        rec['status'] = 'failed'
        rec['error'] = str(exc)[-2000:]
        print('[' + rn + '] EXPORT FAILED: ' + rec['error'], flush=True)
        notify('export failed', rn + ': ' + rec['error'][-1500:], priority='high', tags=['warning'])
    finally:
        if not KEEP_MERGED:
            shutil.rmtree(merged, ignore_errors=True)
        rec_file = EDGE_DIR / 'exports' / (rn + '.json')
        rec_file.parent.mkdir(parents=True, exist_ok=True)
        rec_file.write_text(json.dumps(rec, indent=2, default=str), encoding='utf-8')
        if rec['status'] in ('ok', 'failed'):
            hub_upload_file(rec_file, EDGE_PREFIX + '/exports/' + rn + '.json', '[edge-eval] export record ' + rn)
    return rec


for e in EXP:
    rn = e['run_name']
    if rn not in NEEDED_QUANTS:
        continue
    if minutes_left() < EXPORT_MINUTES_ESTIMATE:
        print('[' + rn + '] Not exporting: ' + format(minutes_left(), '.0f') + ' min left before the reserve.')
        EXPORT_RECORDS[rn] = {'run_name': rn, 'model': e['model'], 'status': 'skipped_for_time', 'quants': {}}
        continue
    print(LF + '#' * 72 + LF + '# Export ' + rn + ': ' + ', '.join(sorted(NEEDED_QUANTS[rn], key=_qkey))
          + LF + '#' * 72, flush=True)
    export_run(e, NEEDED_QUANTS[rn])

print()
for rn, rec in EXPORT_RECORDS.items():
    print(rn.ljust(30) + str(rec.get('status')).ljust(18) + ', '.join(sorted(rec.get('quants', {}), key=_qkey)))

In [ ]:
# 9. Prompt helpers, llama-server client, throughput (llama-bench) and memory footprint
import requests
from src.data.dataset import fit_prompt_to_budget
from src.data.prompts import extract_cypher, strip_leading_bos, template_supports_system
from src.data.schema_filter import apply_schema_mode
from src.inference.predict import prompt_reserve_tokens
from src.training.model_loader import load_tokenizer

# --------------------------------------------------------------------------- #
# Prompts: identical construction to app.py predict (history/030.md)
# --------------------------------------------------------------------------- #
_TOKENIZERS = {}


def edge_config(config_path, prompt_mode):
    cfg = load_config(config_path)
    cfg.data.schema_mode = prompt_mode
    return cfg


def get_tokenizer(cfg):
    key = cfg.model.key
    if key not in _TOKENIZERS:
        _TOKENIZERS[key] = load_tokenizer(cfg, padding_side='left')
    return _TOKENIZERS[key]


def build_prompts(cfg, records):
    tok = get_tokenizer(cfg)
    supports_system = cfg.model.spec.supports_system_role
    if supports_system is None:
        supports_system = template_supports_system(tok)
    reserve = prompt_reserve_tokens(cfg)
    fitted = []
    for rec in records:
        question = rec.get('question') or ''
        schema_text = apply_schema_mode(
            rec.get('schema') or '', question, cfg.data.schema_mode,
            min_nodes=cfg.data.filter_min_nodes,
            keep_patterns_for_kept_nodes=cfg.data.filter_keep_all_patterns_for_kept_nodes,
            similarity_threshold=cfg.data.similarity_threshold,
            similarity_model=cfg.data.similarity_model,
        )
        fitted.append(fit_prompt_to_budget(tok, question, schema_text, cfg, supports_system, reserve_tokens=reserve))
    return tok, fitted, reserve


def stop_strings(cfg, tok):
    stop = [] if EDGE_STOP_MODE == 'hf' else list(cfg.generation.stop_strings)
    eos = getattr(tok, 'eos_token', None)
    if eos and eos not in stop:
        stop.append(eos)
    return stop


# --------------------------------------------------------------------------- #
# llama-server client (records load time, memory and per-request timings)
# --------------------------------------------------------------------------- #
_PORT = {'next': 8790}


class LlamaServer:
    def __init__(self, gguf, threads, ctx, log_path):
        self.gguf, self.threads, self.ctx = str(gguf), int(threads), int(ctx)
        self.log_path = Path(log_path)
        self.port = _PORT['next']
        _PORT['next'] += 1
        self.proc, self._fh, self.url = None, None, None
        self.load_s, self.rss_after_load_mb = None, None
        self.session = requests.Session()

    def start(self, timeout_s=900):
        cmd = [str(LLAMA_BIN / 'llama-server'), '-m', self.gguf, '-c', str(self.ctx), '-t', str(self.threads),
               '-tb', str(self.threads), '--host', '127.0.0.1', '--port', str(self.port), '--parallel', '1',
               '--no-mmap', '--no-warmup']
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self._fh = open(self.log_path, 'w', encoding='utf-8')
        self._fh.write('$ ' + ' '.join(cmd) + LF)
        self._fh.flush()
        t0 = time.perf_counter()
        self.proc = subprocess.Popen(cmd, stdout=self._fh, stderr=subprocess.STDOUT, start_new_session=True)
        url = 'http://127.0.0.1:' + str(self.port)
        while time.perf_counter() - t0 < timeout_s:
            if self.proc.poll() is not None:
                self._close_log()
                tail = LF.join(self.log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-20:])
                raise RuntimeError('llama-server exited with code ' + str(self.proc.returncode)
                                   + ' while loading:' + LF + tail)
            try:
                r = self.session.get(url + '/health', timeout=5)
                if r.status_code == 200:
                    self.load_s = time.perf_counter() - t0
                    self.url = url
                    self.rss_after_load_mb = self.rss_mb('VmRSS')
                    return self
            except requests.RequestException:
                pass
            time.sleep(0.25)
        self.stop()
        raise RuntimeError('llama-server did not become healthy within ' + str(timeout_s) + ' s.')

    def rss_mb(self, field='VmRSS'):
        try:
            for line in Path('/proc/' + str(self.proc.pid) + '/status').read_text().splitlines():
                if line.startswith(field + ':'):
                    return round(int(line.split()[1]) / 1024, 1)
        except Exception:
            return None
        return None

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def complete(self, prompt, n_predict, stop, timeout_s=REQUEST_TIMEOUT_S):
        payload = {'prompt': prompt, 'n_predict': int(n_predict), 'temperature': 0.0, 'top_k': 1, 'top_p': 1.0,
                   'stop': stop, 'cache_prompt': CACHE_PROMPT, 'stream': False}
        r = self.session.post(self.url + '/completion', json=payload, timeout=timeout_s)
        r.raise_for_status()
        return r.json()

    def _close_log(self):
        if self._fh is not None:
            try:
                self._fh.close()
            except Exception:
                pass
            self._fh = None

    def stop(self):
        if self.proc is not None and self.proc.poll() is None:
            try:
                os.killpg(self.proc.pid, signal.SIGTERM)
                self.proc.wait(timeout=30)
            except Exception:
                try:
                    self.proc.kill()
                    self.proc.wait(timeout=10)
                except Exception:
                    pass
        self._close_log()


def stopped_at_limit(resp):
    return bool(resp.get('stopped_limit')) or resp.get('stop_type') == 'limit'


# --------------------------------------------------------------------------- #
# llama-bench (synthetic ppN / tgN throughput) and footprint
# --------------------------------------------------------------------------- #
BENCH_THREADS = sorted({t for t in THREAD_SWEEP if 1 <= t <= LOGICAL_CPUS} | {LOGICAL_CPUS})
BENCH_KEYS = ('n_prompt', 'n_gen', 'n_threads', 'n_batch', 'n_ubatch', 'avg_ns', 'stddev_ns', 'avg_ts', 'stddev_ts',
              'model_size', 'model_n_params', 'model_type', 'cpu_info', 'build_commit', 'type_k', 'type_v',
              'test_time')


def run_llama_bench(gguf, threads, log_path):
    cmd = [str(LLAMA_BIN / 'llama-bench'), '-m', str(gguf), '-p', str(BENCH_PROMPT_TOKENS),
           '-n', str(BENCH_GEN_TOKENS), '-t', ','.join(str(t) for t in threads), '-r', str(BENCH_REPS), '-o', 'json']
    print('$ ' + ' '.join(cmd), flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=BENCH_TIMEOUT_S)
    Path(log_path).write_text('$ ' + ' '.join(cmd) + LF + proc.stderr + LF + '# stdout' + LF + proc.stdout,
                              encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError('llama-bench exited ' + str(proc.returncode) + ': ' + proc.stderr[-800:])
    out = proc.stdout
    data = json.loads(out[out.find('['):out.rfind(']') + 1])
    return data, round(time.time() - t0, 1)


def choose_threads(rows, prompt_tokens, gen_tokens):
    by_t = {}
    for r in rows:
        if r.get('error') or r.get('quant') != ACCURACY_QUANT:
            continue
        t = int(r['n_threads'])
        if r.get('n_prompt') and not r.get('n_gen'):
            by_t.setdefault(t, {})['pp'] = r['avg_ts']
        elif r.get('n_gen') and not r.get('n_prompt'):
            by_t.setdefault(t, {})['tg'] = r['avg_ts']
    est = {t: prompt_tokens / v['pp'] + gen_tokens / v['tg'] for t, v in by_t.items() if v.get('pp') and v.get('tg')}
    if not est:
        return LOGICAL_CPUS, {}
    return min(est, key=est.get), est


def measure_footprint(gguf, threads, ctx, prompt, stop, log_path):
    srv = LlamaServer(gguf, threads, ctx, log_path)
    try:
        srv.start()
        t0 = time.perf_counter()
        resp = srv.complete(prompt, 32, stop)
        first_ms = (time.perf_counter() - t0) * 1000
        tm = resp.get('timings') or {}
        return {'threads': threads, 'load_s': round(srv.load_s, 2), 'rss_after_load_mb': srv.rss_after_load_mb,
                'peak_rss_mb': srv.rss_mb('VmHWM'), 'first_request_ms': round(first_ms, 1),
                'first_prompt_tokens': resp.get('tokens_evaluated') or tm.get('prompt_n'),
                'first_prompt_ms': tm.get('prompt_ms'), 'first_predicted_n': tm.get('predicted_n')}
    finally:
        srv.stop()


for model in PENDING_BENCH:
    e = BENCH_PLAN[model]
    rn = e['run_name']
    ex = EXPORT_RECORDS.get(rn) or {}
    if ex.get('status') != 'ok':
        BENCH_RESULTS[model] = {'model': model, 'run_name': rn, 'status': 'export_' + str(ex.get('status')),
                                'error': ex.get('error'), **_provenance()}
        print('[bench ' + model + '] skipped: export status ' + str(ex.get('status')))
        continue
    print(LF + '#' * 72 + LF + '# Benchmark ' + model + ' (' + rn + ')' + LF + '#' * 72, flush=True)
    cfg = edge_config(e['config'], e['schema_mode'])
    tok, fitted, _reserve = build_prompts(cfg, EDGE_RECORDS)
    lens = sorted(len(f.prompt_ids) for f in fitted)
    typical_prompt = lens[len(lens) // 2]
    median_prompt = strip_leading_bos(tok, fitted[sorted(range(len(fitted)),
                                                         key=lambda i: len(fitted[i].prompt_ids))[len(fitted) // 2]].prompt)
    ctx = cfg.data.max_seq_length + cfg.generation.max_new_tokens + 64
    stop = stop_strings(cfg, tok)

    bench_rows = []
    for q in sorted(set(LATENCY_QUANTS) | {ACCURACY_QUANT}, key=_qkey):
        info = ex['quants'].get(q)
        if not info or not Path(info['path']).exists():
            continue
        if minutes_left() < 20:
            print('[bench ' + model + '] stopping: too little session time left.')
            break
        threads = BENCH_THREADS if q in THREAD_SWEEP_QUANTS else [LOGICAL_CPUS]
        try:
            data, secs = run_llama_bench(info['path'], threads, LOG_DIR / ('bench-' + model + '-' + q.lower() + '.log'))
        except Exception as exc:
            bench_rows.append({'model': model, 'run_name': rn, 'quant': q, 'error': str(exc)[-800:]})
            print('[bench ' + model + ' ' + q + '] FAILED: ' + str(exc)[-400:])
            continue
        for d in data:
            bench_rows.append({'model': model, 'run_name': rn, 'quant': q, 'gguf_size_bytes': info['size_bytes'],
                               'bench_wall_s': secs, **{k: d.get(k) for k in BENCH_KEYS}})
        for d in data:
            kind = 'pp' if d.get('n_prompt') else 'tg'
            print('  ' + q.ljust(8) + kind + str(d.get('n_prompt') or d.get('n_gen')).ljust(5) + ' t='
                  + str(d.get('n_threads')).ljust(3) + format(d.get('avg_ts') or 0, '8.2f') + ' tok/s')

    chosen, estimates = choose_threads(bench_rows, typical_prompt, TYPICAL_GEN_TOKENS)
    if isinstance(SERVER_THREADS, int):
        chosen = SERVER_THREADS

    footprint = []
    for q in sorted(LATENCY_QUANTS, key=_qkey):
        info = ex['quants'].get(q)
        if not info or not Path(info['path']).exists() or minutes_left() < 10:
            continue
        try:
            fp = measure_footprint(info['path'], chosen, ctx, median_prompt, stop,
                                   LOG_DIR / ('server-footprint-' + model + '-' + q.lower() + '.log'))
            fp.update(model=model, run_name=rn, quant=q, gguf_size_bytes=info['size_bytes'])
            footprint.append(fp)
            print('  footprint ' + q.ljust(8) + 'load ' + format(fp['load_s'], '.1f') + ' s, peak RSS '
                  + str(fp['peak_rss_mb']) + ' MB, first request ' + format(fp['first_request_ms'] / 1000, '.1f') + ' s')
        except Exception as exc:
            footprint.append({'model': model, 'run_name': rn, 'quant': q, 'error': str(exc)[-800:]})

    rec = {'model': model, 'run_name': rn, 'status': 'completed', 'bench_rows': bench_rows, 'footprint': footprint,
           'bench_threads': BENCH_THREADS, 'threads_chosen': chosen,
           'thread_estimates_s': {str(k): round(v, 2) for k, v in estimates.items()},
           'est_s_per_item': round(estimates.get(chosen, 0.0), 2) or None,
           'typical_prompt_tokens': typical_prompt, 'typical_gen_tokens': TYPICAL_GEN_TOKENS,
           'bench_prompt_tokens': BENCH_PROMPT_TOKENS, 'bench_gen_tokens': BENCH_GEN_TOKENS,
           'bench_reps': BENCH_REPS, 'server_ctx': ctx, **_provenance()}
    bdir = EDGE_DIR / 'bench' / model
    bdir.mkdir(parents=True, exist_ok=True)
    (bdir / 'bench.json').write_text(json.dumps(rec, indent=2, default=str), encoding='utf-8')
    hub_upload_folder(bdir, EDGE_PREFIX + '/bench/' + model, '[edge-eval] bench ' + model)
    BENCH_RESULTS[model] = rec
    print('[bench ' + model + '] threads chosen for serving: ' + str(chosen) + ' (estimates ' + str(rec['thread_estimates_s']) + ')')

THREADS_FOR = {}
for model in BENCH_PLAN:
    if isinstance(SERVER_THREADS, int):
        THREADS_FOR[model] = SERVER_THREADS
    else:
        THREADS_FOR[model] = int((BENCH_RESULTS.get(model) or {}).get('threads_chosen') or LOGICAL_CPUS)
print('Serving threads: ' + str(THREADS_FOR))

In [ ]:
# 10. Accuracy and latency jobs on the CPU: llama-server -> predictions -> app.py evaluate -> upload
EST_S_PER_ITEM = {}


def _s_per_item(job):
    key = (job['model'], job['quant'])
    if key in EST_S_PER_ITEM:
        return EST_S_PER_ITEM[key]
    base = (BENCH_RESULTS.get(job['model']) or {}).get('est_s_per_item')
    if base:
        return base * (1.0 if job['quant'] == ACCURACY_QUANT else 1.6)
    return DEFAULT_S_PER_ITEM


def run_edge_job(job):
    jid, rn, q = job['job_id'], job['run_name'], job['quant']
    ex = EXPORT_RECORDS.get(rn) or {}
    info = (ex.get('quants') or {}).get(q)
    jdir = EDGE_DIR / 'jobs' / jid
    if jdir.exists():
        shutil.rmtree(jdir)
    jdir.mkdir(parents=True)

    cfg = edge_config(job['config'], job['prompt_schema_mode'])
    tok, fitted, reserve = build_prompts(cfg, EDGE_RECORDS)
    stop = stop_strings(cfg, tok)
    threads = THREADS_FOR.get(job['model'], LOGICAL_CPUS)
    ctx = cfg.data.max_seq_length + cfg.generation.max_new_tokens + 64
    max_new = cfg.generation.max_new_tokens
    started = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    t_job = time.time()

    rows = []
    srv = LlamaServer(info['path'], threads, ctx, jdir / 'server.log')
    try:
        srv.start()
        print('[' + jid + '] server ready in ' + format(srv.load_s, '.1f') + ' s, RSS ' + str(srv.rss_after_load_mb)
              + ' MB, threads ' + str(threads), flush=True)
        tw = time.perf_counter()
        srv.complete(strip_leading_bos(tok, fitted[0].prompt), 1, stop)  # untimed warm-up
        warmup_ms = (time.perf_counter() - tw) * 1000
        t_loop = time.time()
        for i, (rec, f) in enumerate(zip(EDGE_RECORDS, fitted), 1):
            prompt = strip_leading_bos(tok, f.prompt)
            resp, err, attempts = {}, None, 0
            t0 = time.perf_counter()
            for attempts in range(1, 4):
                try:
                    resp = srv.complete(prompt, max_new, stop)
                    err = None
                    break
                except Exception as exc:
                    err = type(exc).__name__ + ': ' + str(exc)[:300]
                    if not srv.alive():
                        raise RuntimeError('llama-server died during the job: ' + err)
                    time.sleep(2 * attempts)
            wall_ms = (time.perf_counter() - t0) * 1000
            raw = resp.get('content', '') if resp else ''
            tm = resp.get('timings') or {}
            rows.append({
                'instance_id': rec.get('instance_id'), 'question': rec.get('question'),
                'gold_cypher': (rec.get('cypher') or '').strip(), 'predicted_raw': raw,
                'predicted_cypher': extract_cypher(raw),
                'database_reference': rec.get('database_reference'), 'data_source': rec.get('data_source'),
                'schema_mode': cfg.data.schema_mode, 'schema_chars': len(f.schema_text),
                'schema_chars_untrimmed': f.schema_chars_untrimmed, 'schema_trimmed': f.schema_trimmed,
                'prompt_over_budget': f.over_budget, 'prompt_tokens': len(f.prompt_ids),
                'edge_wall_ms': round(wall_ms, 1), 'edge_attempts': attempts, 'edge_request_error': err,
                'edge_tokens_evaluated': resp.get('tokens_evaluated'), 'edge_tokens_cached': resp.get('tokens_cached'),
                'edge_prompt_n': tm.get('prompt_n'), 'edge_prompt_ms': tm.get('prompt_ms'),
                'edge_prompt_tps': tm.get('prompt_per_second'), 'edge_predicted_n': tm.get('predicted_n'),
                'edge_predicted_ms': tm.get('predicted_ms'), 'edge_predicted_tps': tm.get('predicted_per_second'),
                'edge_stopped_limit': stopped_at_limit(resp) if resp else None,
                'edge_truncated': resp.get('truncated') if resp else None,
            })
            if i % 10 == 0 or i == len(EDGE_RECORDS):
                rate = (time.time() - t_loop) / i
                print('[' + jid + '] ' + str(i) + '/' + str(len(EDGE_RECORDS)) + ' | ' + format(rate, '.1f')
                      + ' s/item | ETA ' + format(rate * (len(EDGE_RECORDS) - i) / 60, '.0f') + ' min', flush=True)
        peak_rss = srv.rss_mb('VmHWM')
        load_s, rss_load = srv.load_s, srv.rss_after_load_mb
    finally:
        srv.stop()

    elapsed = time.time() - t_loop
    pred = jdir / 'predictions.jsonl'
    with pred.open('w', encoding='utf-8') as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False, default=str) + LF)
    meta = {'run_name': cfg.run_name, 'tag': jid, 'model_key': cfg.model.key, 'model_path': cfg.model.resolved_path(),
            'adapter': str(job['adapter_dir']), 'gguf': info['path'], 'backend': 'llamacpp',
            'backend_info': {'threads': threads, 'n_ctx': ctx, 'cache_prompt': CACHE_PROMPT, 'stop': stop,
                             'llama_cpp_commit': LLAMA_COMMIT},
            'split': SPLIT, 'n_instances': len(rows), 'schema_mode': cfg.data.schema_mode,
            'max_seq_length': cfg.data.max_seq_length, 'prompt_reserve_tokens': reserve,
            'n_schema_trimmed': sum(1 for f in fitted if f.schema_trimmed),
            'n_prompt_over_budget': sum(1 for f in fitted if f.over_budget),
            'max_new_tokens': max_new, 'temperature': 0.0, 'batch_size': 1,
            'total_seconds': round(elapsed, 2), 'seconds_per_item': round(elapsed / max(1, len(rows)), 3),
            'mean_prompt_tokens': round(sum(len(f.prompt_ids) for f in fitted) / max(1, len(fitted)), 1),
            'host': platform.node(), 'platform': platform.platform(),
            'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S')}
    (jdir / 'predictions.meta.json').write_text(json.dumps(meta, indent=2, default=str), encoding='utf-8')

    flags = ((' --execute' if RUN_EXECUTION else '') + (' --validate-syntax' if RUN_SYNTAX_CHECK else '')
             + ' --dead-db-policy ' + DEAD_DB_POLICY)
    run_step('evaluate-' + jid, 'python app.py evaluate --predictions ' + str(pred) + flags + ' --out '
             + str(jdir / 'report.md'), log_path=jdir / 'evaluate.log', echo=False)
    for required in ('predictions.scored.jsonl', 'report.json'):
        if not (jdir / required).exists():
            raise RuntimeError(jid + ': missing ' + required + ' after evaluation.')

    EST_S_PER_ITEM[(job['model'], q)] = meta['seconds_per_item']
    rec = _job_record(
        job, status='completed', started=started,
        finished=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        wall_s=round(time.time() - t_job, 1), n_rows=len(rows), seconds_per_item=meta['seconds_per_item'],
        generation_seconds=meta['total_seconds'], threads=threads, server_ctx=ctx,
        server_load_s=round(load_s, 2), rss_after_load_mb=rss_load, peak_rss_mb=peak_rss,
        warmup_ms=round(warmup_ms, 1), gguf_file=info['file'], gguf_size_bytes=info['size_bytes'],
        gguf_sha256=info['sha256'], stop_strings=stop, edge_stop_mode=EDGE_STOP_MODE, cache_prompt=CACHE_PROMPT,
        prompt_reserve_tokens=reserve, n_schema_trimmed=meta['n_schema_trimmed'],
        n_prompt_over_budget=meta['n_prompt_over_budget'],
        n_request_errors=sum(1 for r in rows if r['edge_request_error']),
        n_stopped_limit=sum(1 for r in rows if r['edge_stopped_limit']),
        merge_adapter_verified=ex.get('adapter_verified'),
        merge_model_class=(ex.get('merge_metadata') or {}).get('model_class'),
        python=platform.python_version(),
    )
    (jdir / 'job.json').write_text(json.dumps(rec, indent=2, default=str), encoding='utf-8')
    rec['uploaded'] = hub_upload_folder(jdir, EDGE_PREFIX + '/jobs/' + jid, '[edge-eval] ' + EDGE_TAG + ' ' + jid)
    if UPLOAD_TO_HUB and not rec['uploaded']:
        print('WARNING: ' + jid + ' finished but its upload failed; it is only kept in this session.')
    return rec


for i, job in enumerate(PENDING_JOBS, 1):
    jid = job['job_id']
    info = ((EXPORT_RECORDS.get(job['run_name']) or {}).get('quants') or {}).get(job['quant'])
    if not info or not Path(info['path']).exists():
        status = (EXPORT_RECORDS.get(job['run_name']) or {}).get('status', 'not_exported')
        JOB_RESULTS[jid] = _job_record(job, status='export_' + str(status),
                                       error=(EXPORT_RECORDS.get(job['run_name']) or {}).get('error'))
        print('[' + jid + '] not run: no local ' + job['quant'] + ' GGUF (export ' + str(status) + ').')
        continue
    need = len(EDGE_RECORDS) * _s_per_item(job) / 60 + 5
    if minutes_left() < need:
        SKIPPED_FOR_TIME.append(jid)
        JOB_RESULTS[jid] = _job_record(job, status='not_started',
                                       error='needs ~' + format(need, '.0f') + ' min, '
                                             + format(minutes_left(), '.0f') + ' min left')
        print('[' + jid + '] not started: needs ~' + format(need, '.0f') + ' min.')
        continue
    print(LF + '#' * 72 + LF + '# Job ' + str(i) + '/' + str(len(PENDING_JOBS)) + ': ' + jid + ' (~'
          + format(need, '.0f') + ' min)' + LF + '#' * 72, flush=True)
    try:
        JOB_RESULTS[jid] = run_edge_job(job)
        STATUS['jobs'][jid] = 'completed'
    except KeyboardInterrupt:
        raise
    except Exception as exc:
        STATUS['jobs'][jid] = 'failed'
        JOB_RESULTS[jid] = _job_record(job, status='failed', error=str(exc)[-2000:])
        print('[' + jid + '] FAILED: ' + str(exc), flush=True)
        notify('job failed', jid + ': ' + str(exc)[-1500:], priority='high', tags=['warning'])

done = [j['job_id'] for j in JOBS if JOB_RESULTS.get(j['job_id'], {}).get('status') == 'completed']
not_done = [(j['job_id'], JOB_RESULTS.get(j['job_id'], {}).get('status', 'not_started')) for j in JOBS
            if JOB_RESULTS.get(j['job_id'], {}).get('status') != 'completed']
print()
print('Expected: ' + str(len(JOBS)) + ' | Completed: ' + str(len(done)) + ' | Not completed: ' + str(len(not_done)))
for jid, status in not_done:
    print('  ' + status + ': ' + jid)
notify('jobs finished', 'Completed ' + str(len(done)) + '/' + str(len(JOBS)) + '. Session ' + SESSION + '.',
       priority='high' if not_done else 'default', tags=['warning'] if not_done else ['white_check_mark'])

In [ ]:
# 11. Shared statistics, row preparation, and supplementary analyses (GPU reference + CPU jobs)
import math
import re
from statistics import NormalDist

import numpy as np
from IPython.display import display
from src.data.schema_filter import parse_schema
from src.evaluation.metrics import _ngram_counts, normalise_cypher, tokenize_cypher

Z = NormalDist().inv_cdf(0.5 + CI_LEVEL / 2)
ALPHA = 1 - CI_LEVEL
RNG = np.random.default_rng(STATS_SEED)
JUDGED_STATUSES = ('compared', 'pred_failed')
FUNCTIONAL_SOURCE = 'neo4jLabs_functional_cypher'
NAN = float('nan')


def wilson(k, n):
    if n == 0:
        return NAN, NAN
    p = k / n
    denom = 1 + Z * Z / n
    centre = (p + Z * Z / (2 * n)) / denom
    half = Z * math.sqrt(p * (1 - p) / n + Z * Z / (4 * n * n)) / denom
    return min(max(0.0, centre - half), p), max(min(1.0, centre + half), p)


def mcnemar_exact(b, c):
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)


def paired_diff_ci(a_vals, b_vals):
    d = np.asarray(b_vals, float) - np.asarray(a_vals, float)
    if len(d) == 0:
        return {}
    means = d[RNG.integers(0, len(d), size=(BOOTSTRAP_SAMPLES, len(d)))].mean(axis=1)
    lo, hi = np.quantile(means, [ALPHA / 2, 1 - ALPHA / 2])
    lo90, hi90 = np.quantile(means, [ALPHA, 1 - ALPHA])
    return {'difference_b_minus_a': float(d.mean()), 'diff_ci_low': float(lo), 'diff_ci_high': float(hi),
            'tost_ci_low': float(lo90), 'tost_ci_high': float(hi90),
            'equivalent_within_margin': bool(lo90 > -EQUIV_MARGIN and hi90 < EQUIV_MARGIN),
            'bootstrap_p': min(1.0, 2 * min(float((means <= 0).mean()), float((means >= 0).mean())))}


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        adj[i] = running
    return adj


def _gleu_counts(ref, hyp):
    rc = _ngram_counts(tokenize_cypher(ref), 1, 4)
    hc = _ngram_counts(tokenize_cypher(hyp), 1, 4)
    return sum((rc & hc).values()), sum(hc.values()), sum(rc.values())


def _num(df, col):
    if df is None or len(df) == 0 or col not in df.columns:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[col], errors='coerce').dropna()


def prep_rows(df):
    df = df.copy()
    for col in ('execution_status', 'gold_executed_ok', 'pred_executed_ok', 'execution_match', 'syntax_valid',
                'gold_row_count', 'pred_row_count', 'exact_match', 'normalised_exact_match', 'gleu',
                'database_reference', 'data_source', 'complexity', 'question', 'predicted_raw'):
        if col not in df.columns:
            df[col] = None
    df['instance_id'] = df['instance_id'].astype(str)
    df['gold_cypher'] = df['gold_cypher'].fillna('').astype(str)
    df['predicted_cypher'] = df['predicted_cypher'].fillna('').astype(str)
    df['has_database'] = df['database_reference'].fillna('').astype(str).str.strip().ne('')
    counts = [_gleu_counts(g, p) for g, p in zip(df['gold_cypher'], df['predicted_cypher'])]
    df['gleu_overlap'] = [c[0] for c in counts]
    df['gleu_hyp_ngrams'] = [c[1] for c in counts]
    df['gleu_ref_ngrams'] = [c[2] for c in counts]
    df['empty_output'] = df['predicted_cypher'].str.strip().eq('')
    df['judged'] = df['execution_status'].isin(JUDGED_STATUSES) & df['gold_executed_ok'].eq(True)
    df['correct'] = df['judged'] & df['execution_match'].eq(True)
    df['gold_rows_n'] = pd.to_numeric(df['gold_row_count'], errors='coerce')
    df['pred_rows_n'] = pd.to_numeric(df['pred_row_count'], errors='coerce')
    return df


def bootstrap_gleu(df):
    n = len(df)
    if n == 0:
        return NAN, NAN
    o, h, r = (df[c].to_numpy(float) for c in ('gleu_overlap', 'gleu_hyp_ngrams', 'gleu_ref_ngrams'))
    idx = RNG.integers(0, n, size=(BOOTSTRAP_SAMPLES, n))
    so, sh, sr = o[idx].sum(axis=1), h[idx].sum(axis=1), r[idx].sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        vals = np.nan_to_num(np.minimum(so / sh, so / sr))
    lo, hi = np.quantile(vals, [ALPHA / 2, 1 - ALPHA / 2])
    return float(lo), float(hi)


def job_metrics(df):
    n = len(df)
    out = {'n': n}
    if n == 0:
        return out
    o, h, r = df['gleu_overlap'].sum(), df['gleu_hyp_ngrams'].sum(), df['gleu_ref_ngrams'].sum()
    out['google_bleu_corpus'] = min(o / h, o / r) if h and r else 0.0
    out['google_bleu_corpus_ci_low'], out['google_bleu_corpus_ci_high'] = bootstrap_gleu(df)
    for name in ('exact_match', 'normalised_exact_match'):
        k = int(df[name].eq(True).sum())
        out[name] = k / n
        out[name + '_ci_low'], out[name + '_ci_high'] = wilson(k, n)
    out['empty_output_rate'] = float(df['empty_output'].mean())
    out['n_has_database'] = int(df['has_database'].sum())
    out['n_connection_failed'] = int(df['execution_status'].eq('connection_failed').sum())
    j = df[df['judged']]
    out['n_judged'] = len(j)
    if len(j):
        k = int(j['correct'].sum())
        out['execution_accuracy'] = k / len(j)
        out['execution_accuracy_ci_low'], out['execution_accuracy_ci_high'] = wilson(k, len(j))
        out['executable_rate_gold_ok'] = float(j['pred_executed_ok'].eq(True).mean())
        nonempty = j[j['gold_rows_n'].fillna(0) > 0]
        out['n_judged_nonempty_gold'] = len(nonempty)
        if len(nonempty):
            out['execution_accuracy_nonempty_gold'] = float(nonempty['correct'].mean())
        out['share_correct_with_empty_gold'] = float((j['correct'] & j['gold_rows_n'].fillna(0).eq(0)).sum()
                                                     / max(1, k))
    validated = df[df['syntax_valid'].notna()]
    if len(validated):
        out['syntax_valid_rate'] = float(validated['syntax_valid'].eq(True).mean())
    return out


# --------------------------------------------------------------------------- #
# Every job with scored rows: GPU references and CPU jobs
# --------------------------------------------------------------------------- #
ALL_JOBS = {}
for jid, v in REF.items():
    ALL_JOBS[jid] = {'source': 'gpu_reference', 'job': v['job'], 'rows': prep_rows(v['rows'])}
for jid, rec in JOB_RESULTS.items():
    scored = EDGE_DIR / 'jobs' / jid / 'predictions.scored.jsonl'
    if rec.get('status') != 'completed' or not scored.exists():
        continue
    df = pd.DataFrame([json.loads(ln) for ln in scored.read_text(encoding='utf-8').splitlines() if ln.strip()])
    ALL_JOBS[jid] = {'source': 'edge_cpu', 'job': rec, 'rows': prep_rows(df)}
EDGE_JOB_IDS = [jid for jid, v in ALL_JOBS.items() if v['source'] == 'edge_cpu']
print('Jobs with scored rows: ' + str(len(ALL_JOBS)) + ' (' + str(len(EDGE_JOB_IDS)) + ' on the CPU)')

# --------------------------------------------------------------------------- #
# Cypher element extraction (heuristic; documented in the README)
# --------------------------------------------------------------------------- #
_STR_RE = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"")
_NODE_LABELS_RE = re.compile(r'\(\s*(?:[A-Za-z_]\w*)?\s*:\s*([^(){}\[\]]+?)\s*(?=\{|\))')
_REL_TYPES_RE = re.compile(r'\[\s*(?:[A-Za-z_]\w*)?\s*:\s*([^\[\]{}*]+?)\s*(?=\*|\{|\])')
_PROP_ACCESS_RE = re.compile(r'\b([A-Za-z_]\w*)\s*\.\s*`?([A-Za-z_]\w*)\b`?(?!\s*[(.])')
_PATTERN_MAP_RE = re.compile(r'[(\[]\s*(?:[A-Za-z_]\w*)?\s*(?::[^(){}\[\]]*)?\{([^{}]*)\}\s*[)\]]')
_PROJ_RE = re.compile(r'[{,]\s*\.\s*([A-Za-z_]\w*)')
_NAMESPACES = {'apoc', 'gds', 'db', 'dbms', 'point', 'date', 'datetime', 'localdatetime', 'localtime', 'time',
               'duration', 'math', 'text', 'coll', 'spatial', 'genai', 'vector'}


def _split_names(text):
    out = set()
    for part in re.split(r'[:|&]', text):
        name = part.strip().strip('`').lstrip('!').strip()
        if re.fullmatch(r'[A-Za-z_]\w*', name):
            out.add(name)
    return out


def cypher_elements(query):
    q = _STR_RE.sub("''", query or '')
    labels, rels, props = set(), set(), set()
    for m in _NODE_LABELS_RE.finditer(q):
        labels |= _split_names(m.group(1))
    for m in _REL_TYPES_RE.finditer(q):
        rels |= _split_names(m.group(1))
    bound = (set(re.findall(r'\(\s*([A-Za-z_]\w*)', q)) | set(re.findall(r'\[\s*([A-Za-z_]\w*)\s*[:\]*{]', q))
             | set(re.findall(r'(?i)\bAS\s+([A-Za-z_]\w*)', q)))
    for m in _PROP_ACCESS_RE.finditer(q):
        if m.group(1) in bound and m.group(1).lower() not in _NAMESPACES:
            props.add(m.group(2))
    for m in _PATTERN_MAP_RE.finditer(q):
        props |= set(re.findall(r'([A-Za-z_]\w*)\s*:', m.group(1)))
    props |= set(_PROJ_RE.findall(q))
    return labels, rels, props


def schema_elements(text):
    s = parse_schema(text or '')
    if not s.parsed:
        return None
    labels = set(s.nodes) | {a for a, _r, _b in s.patterns if a} | {b for _a, _r, b in s.patterns if b}
    rels = set(s.rel_props) | {r for _a, r, _b in s.patterns}
    props = set()
    for group in list(s.nodes.values()) + list(s.rel_props.values()):
        props |= set(group)
    return labels, rels, props


FULL_EL = {iid: schema_elements(r.get('schema') or '') for iid, r in REF_BY_ID.items()}
_PROMPT_EL = {}


def prompt_elements(iid, mode):
    key = (iid, mode)
    if key not in _PROMPT_EL:
        full = FULL_EL.get(iid)
        if full is None or mode == 'enhanced':
            _PROMPT_EL[key] = full
        else:
            rec = REF_BY_ID[iid]
            text = apply_schema_mode(rec.get('schema') or '', rec.get('question') or '', mode, **FILTER_KW)
            _PROMPT_EL[key] = schema_elements(text) or (set(), set(), set())
    return _PROMPT_EL[key]


def _job_prompt_mode(rec):
    return rec.get('prompt_schema_mode') or rec.get('schema_mode') or 'enhanced'


EMPTY_DF = pd.DataFrame()
SUBSET_METRICS = HALLUCINATION = RESULT_BREAKDOWN = COVERAGE_ROWS = COVERAGE = COVERAGE_EA = EMPTY_DF
AGREEMENT = COMPOSITION = EMPTY_DF

if RUN_SUPPLEMENTARY:
    # (a) Metrics on informative subsets: functional_cypher is near-saturated and inflates text metrics.
    subsets = {
        'all': lambda d: d,
        'excl_functional_cypher': lambda d: d[d['data_source'] != FUNCTIONAL_SOURCE],
        'with_database': lambda d: d[d['has_database']],
        'without_database': lambda d: d[~d['has_database']],
    }
    rows = []
    for jid, v in ALL_JOBS.items():
        rec = v['job']
        for name, fn in subsets.items():
            rows.append({'job_id': jid, 'source': v['source'], 'run_name': rec.get('run_name'),
                         'model': rec.get('model'), 'condition': rec.get('condition'),
                         'prompt_schema_mode': _job_prompt_mode(rec), 'quant': rec.get('quant'),
                         'subset': name, **job_metrics(fn(v['rows']))})
    SUBSET_METRICS = pd.DataFrame(rows)

    # (b) Exact-match filter coverage: are the elements the gold query needs still in the pruned schema?
    cov = []
    for iid, rec in REF_BY_ID.items():
        full = FULL_EL.get(iid)
        gl, gr, gp = cypher_elements(rec.get('cypher') or '')
        row = {'instance_id': iid, 'data_source': rec.get('data_source'),
               'has_database': bool(rec.get('database_reference')), 'schema_parsed': full is not None,
               'in_edge_subset': iid in set(EDGE_IDS)}
        if full is not None:
            pl, pr, pp = prompt_elements(iid, FILTER_MODE)
            fl, fr, fp = full
            known = (gl & fl, gr & fr, gp & fp)
            missing = (known[0] - pl, known[1] - pr, known[2] - pp)
            pruned_text = apply_schema_mode(rec.get('schema') or '', rec.get('question') or '', FILTER_MODE,
                                            **FILTER_KW)
            row.update({
                'gold_labels': len(gl), 'gold_rel_types': len(gr), 'gold_properties': len(gp),
                'gold_unknown_labels': '; '.join(sorted(gl - fl)), 'gold_unknown_rel_types': '; '.join(sorted(gr - fr)),
                'gold_unknown_properties': '; '.join(sorted(gp - fp)),
                'missing_labels': '; '.join(sorted(missing[0])), 'missing_rel_types': '; '.join(sorted(missing[1])),
                'missing_properties': '; '.join(sorted(missing[2])),
                'fully_covered': not any(missing),
                'schema_chars_full': len(rec.get('schema') or ''), 'schema_chars_pruned': len(pruned_text),
                'filter_fell_back': pruned_text.strip() == (rec.get('schema') or '').strip(),
            })
        cov.append(row)
    COVERAGE_ROWS = pd.DataFrame(cov)
    parsed = COVERAGE_ROWS[COVERAGE_ROWS['schema_parsed']].copy()
    parsed['chars_ratio'] = parsed['schema_chars_pruned'] / parsed['schema_chars_full'].replace(0, np.nan)
    cov_rows = []
    for group, sub in [('all', parsed)] + [(src, g) for src, g in parsed.groupby('data_source')]:
        if not len(sub):
            continue
        cov_rows.append({'group': group, 'n_parsed': len(sub),
                         'share_fully_covered': float(sub['fully_covered'].mean()),
                         'share_missing_label': float(sub['missing_labels'].ne('').mean()),
                         'share_missing_rel_type': float(sub['missing_rel_types'].ne('').mean()),
                         'share_missing_property': float(sub['missing_properties'].ne('').mean()),
                         'share_gold_uses_unknown_element': float((sub['gold_unknown_labels'].ne('')
                                                                   | sub['gold_unknown_rel_types'].ne('')
                                                                   | sub['gold_unknown_properties'].ne('')).mean()),
                         'share_filter_fell_back': float(sub['filter_fell_back'].mean()),
                         'median_pruned_chars_ratio': float(sub['chars_ratio'].median())})
    COVERAGE = pd.DataFrame(cov_rows)
    covered = parsed.set_index('instance_id')['fully_covered']

    # Coverage vs execution accuracy (GPU references): does the exact-match loss sit where elements were dropped?
    ea_rows = []
    for m in dict.fromkeys(e['model'] for e in EXP):
        enh, pru = BY_VARIANT.get((m, 'enhanced')), BY_VARIANT.get((m, 'pruned'))
        if not (enh and pru):
            continue
        ids = {'enhanced_trained_enhanced_prompts': enh['run_name'] + '__fine_tuned',
               'exact_match_trained_exact_match_prompts': pru['run_name'] + '__fine_tuned',
               'enhanced_trained_exact_match_prompts': enh['run_name'] + '__fine_tuned__' + pru['schema_mode'] + '_prompts'}
        frames = {k: ALL_JOBS[j]['rows'].drop_duplicates('instance_id').set_index('instance_id')
                  for k, j in ids.items() if j in ALL_JOBS}
        if len(frames) < 2:
            continue
        common = None
        for f in frames.values():
            judged_ids = set(f.index[f['judged']])
            common = judged_ids if common is None else common & judged_ids
        for flag in (True, False):
            group_ids = sorted(i for i in common if covered.get(i) is flag)
            row = {'model': m, 'fully_covered': flag, 'n_judged_all': len(group_ids)}
            for k, f in frames.items():
                row['ea_' + k] = float(f.loc[group_ids, 'correct'].mean()) if group_ids else NAN
            ea_rows.append(row)
    COVERAGE_EA = pd.DataFrame(ea_rows)

    # (c) Schema hallucination: predicted labels / relationship types / properties absent from the schema.
    hal = []
    for jid, v in ALL_JOBS.items():
        rec, df = v['job'], v['rows']
        mode = _job_prompt_mode(rec)
        flags = []
        for _, r in df.iterrows():
            full = FULL_EL.get(r['instance_id'])
            if full is None or not r['predicted_cypher'].strip():
                continue
            pl, pr, pp = cypher_elements(r['predicted_cypher'])
            fl, fr, fp = full
            ql, qr, qp = prompt_elements(r['instance_id'], mode)
            unk = (pl - fl, pr - fr, pp - fp)
            outside = ((pl & fl) - ql) | ((pr & fr) - qr) | ((pp & fp) - qp)
            flags.append({'any': any(unk), 'label': bool(unk[0]), 'rel': bool(unk[1]), 'prop': bool(unk[2]),
                          'outside_prompt': bool(outside), 'judged': bool(r['judged']), 'correct': bool(r['correct'])})
        f = pd.DataFrame(flags)
        if f.empty:
            continue
        hal.append({'job_id': jid, 'source': v['source'], 'run_name': rec.get('run_name'), 'model': rec.get('model'),
                    'condition': rec.get('condition'), 'prompt_schema_mode': mode, 'quant': rec.get('quant'),
                    'n_checked': len(f), 'share_any_unknown_element': float(f['any'].mean()),
                    'share_unknown_label': float(f['label'].mean()), 'share_unknown_rel_type': float(f['rel'].mean()),
                    'share_unknown_property': float(f['prop'].mean()),
                    'share_uses_element_outside_prompt_schema': float(f['outside_prompt'].mean()),
                    'ea_rows_with_unknown': float(f.loc[f['any'] & f['judged'], 'correct'].mean())
                    if (f['any'] & f['judged']).any() else NAN,
                    'ea_rows_without_unknown': float(f.loc[~f['any'] & f['judged'], 'correct'].mean())
                    if (~f['any'] & f['judged']).any() else NAN})
    HALLUCINATION = pd.DataFrame(hal)

    # (d) Execution-result breakdown: matches on empty gold results are weak evidence of a correct query.
    br = []
    for jid, v in ALL_JOBS.items():
        j = v['rows'][v['rows']['judged']]
        if not len(j):
            continue
        g, p = j['gold_rows_n'].fillna(-1), j['pred_rows_n'].fillna(-1)
        cats = np.select(
            [j['correct'] & (g > 0), j['correct'] & (g == 0), ~j['pred_executed_ok'].eq(True),
             (p == 0) & (g > 0), p != g],
            ['correct_nonempty', 'correct_empty_gold', 'pred_failed', 'wrong_pred_empty', 'wrong_row_count'],
            default='wrong_same_row_count')
        counts = pd.Series(cats).value_counts()
        rec = v['job']
        br.append({'job_id': jid, 'source': v['source'], 'model': rec.get('model'), 'condition': rec.get('condition'),
                   'quant': rec.get('quant'), 'n_judged': len(j), 'share_gold_empty': float((g == 0).mean()),
                   **{'n_' + c: int(counts.get(c, 0)) for c in ('correct_nonempty', 'correct_empty_gold', 'pred_failed',
                                                                 'wrong_pred_empty', 'wrong_row_count',
                                                                 'wrong_same_row_count')}})
    RESULT_BREAKDOWN = pd.DataFrame(br)

    # (e) Cross-model agreement among the fine-tuned enhanced GPU references.
    ft = {e['model']: e['run_name'] + '__fine_tuned' for e in EXP if e['variant'] == 'enhanced'}
    ft = {m: j for m, j in ft.items() if j in ALL_JOBS}
    if len(ft) >= 2:
        frames = {m: ALL_JOBS[j]['rows'].drop_duplicates('instance_id').set_index('instance_id') for m, j in ft.items()}
        common = sorted(set.intersection(*[set(f.index[f['judged']]) for f in frames.values()]))
        correct = pd.DataFrame({m: f.loc[common, 'correct'].astype(int) for m, f in frames.items()})
        k = correct.sum(axis=1)
        row = {'models': '; '.join(ft), 'n_judged_all': len(common), 'oracle_ea': float((k > 0).mean()),
               'all_correct': float((k == len(ft)).mean()), 'majority_ea': float((k > len(ft) / 2).mean())}
        for m in ft:
            row['ea_' + m] = float(correct[m].mean())
        for kk in range(len(ft) + 1):
            row['share_correct_by_' + str(kk)] = float((k == kk).mean())
        all_ids = sorted(set.intersection(*[set(f.index) for f in frames.values()]))
        for a, b in [(a, b) for i, a in enumerate(ft) for b in list(ft)[i + 1:]]:
            na = frames[a].loc[all_ids, 'predicted_cypher'].map(normalise_cypher)
            nb = frames[b].loc[all_ids, 'predicted_cypher'].map(normalise_cypher)
            row['identical_norm_' + a + '_vs_' + b] = float((na == nb).mean())
        AGREEMENT = pd.DataFrame([row])

    # (f) Sample composition (explains what the 500-row and edge samples contain).
    base_id = next((j for j in ALL_JOBS if ALL_JOBS[j]['source'] == 'gpu_reference'), None)
    if base_id:
        base = ALL_JOBS[base_id]['rows'].copy()
        base['in_edge_subset'] = base['instance_id'].isin(set(EDGE_IDS))
        COMPOSITION = (base.groupby(['data_source', 'has_database', 'complexity'], dropna=False)
                       .agg(n_reference=('instance_id', 'size'), n_edge=('in_edge_subset', 'sum'))
                       .reset_index().sort_values('n_reference', ascending=False))

    print('Filter coverage (exact-match prompts):')
    display(COVERAGE)
    print('Coverage vs execution accuracy:')
    display(COVERAGE_EA)
    print('Schema hallucination:')
    display(HALLUCINATION)
else:
    print('RUN_SUPPLEMENTARY is False; supplementary tables are empty.')

In [ ]:
# 12. Aggregate the CPU jobs into thesis tables (CIs, paired tests vs GPU, latency, footprint, integrity)
OUT_DIR = REPO_DIR / 'artifacts' / 'thesis-edge' / EDGE_TAG
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
(OUT_DIR / 'tables').mkdir(parents=True)


def _flat(value):
    if isinstance(value, (list, tuple, set)):
        return '; '.join(str(v) for v in value)
    if isinstance(value, dict):
        return json.dumps(value, default=str)
    if isinstance(value, Path):
        return str(value)
    return value


def _g(row, key):
    value = row.get(key) if row is not None else None
    try:
        return float(value) if value is not None and pd.notna(value) else NAN
    except (TypeError, ValueError):
        return NAN


def latency_stats(df):
    out = {}
    for col, name in (('edge_wall_ms', 'wall_ms'), ('edge_prompt_ms', 'prefill_ms'), ('edge_predicted_ms', 'decode_ms')):
        s = _num(df, col)
        if len(s):
            out[name + '_mean'] = float(s.mean())
            out[name + '_p50'] = float(s.quantile(0.5))
            out[name + '_p90'] = float(s.quantile(0.9))
            out[name + '_p95'] = float(s.quantile(0.95))
            out[name + '_max'] = float(s.max())
    pn, gn = _num(df, 'edge_tokens_evaluated'), _num(df, 'edge_predicted_n')
    if len(pn):
        out['prompt_tokens_server_mean'] = float(pn.mean())
    if len(gn):
        out['generated_tokens_mean'] = float(gn.mean())
    pp, tg = _num(df, 'edge_prompt_tps'), _num(df, 'edge_predicted_tps')
    if len(pp):
        out['prefill_tps_median'] = float(pp.median())
    if len(tg):
        out['decode_tps_median'] = float(tg.median())
    wall, pre = _num(df, 'edge_wall_ms'), _num(df, 'edge_prompt_ms')
    if len(wall) and wall.sum() > 0:
        out['prefill_share_of_wall'] = float(pre.sum() / wall.sum()) if len(pre) else NAN
        out['queries_per_hour'] = float(3.6e6 / wall.mean())
    out['n_stopped_limit'] = int(df['edge_stopped_limit'].eq(True).sum()) if 'edge_stopped_limit' in df else 0
    out['n_request_errors'] = int(df['edge_request_error'].notna().sum()) if 'edge_request_error' in df else 0
    return out


def _error_category(r):
    if not str(r.get('predicted_cypher') or '').strip():
        return 'empty_output'
    status = r.get('execution_status')
    if status is None or (isinstance(status, float) and math.isnan(status)):
        return 'not_executed'
    if status in ('connection_failed', 'excluded_dead_db', 'gold_failed'):
        return status
    if r.get('pred_executed_ok') is not True:
        if r.get('syntax_valid') is False:
            return 'invalid_query'
        if r.get('syntax_valid') is True:
            return 'runtime_error'
        return 'execution_error'
    return 'correct' if r.get('execution_match') is True else 'wrong_result'


# --------------------------------------------------------------------------- #
# Per-job overview, with the GPU reference restricted to the same rows
# --------------------------------------------------------------------------- #
overall, long_frames, err_rows = [], [], []
for jid in EDGE_JOB_IDS:
    rec, df = ALL_JOBS[jid]['job'], ALL_JOBS[jid]['rows']
    row = {k: rec.get(k) for k in ('job_id', 'reference_job_id', 'run_name', 'model', 'variant', 'condition', 'quant',
                                   'prompt_schema_mode', 'train_schema_mode', 'threads', 'server_load_s',
                                   'rss_after_load_mb', 'peak_rss_mb', 'warmup_ms', 'seconds_per_item',
                                   'gguf_size_bytes', 'gguf_sha256', 'git_commit', 'llama_cpp_commit', 'cpu_model',
                                   'session')}
    row['job_id'] = jid
    row['gguf_size_mb'] = _g(rec, 'gguf_size_bytes') / 1e6
    row.update(job_metrics(df))
    row.update(latency_stats(df))
    row['prompt_tokens_mean'] = float(_num(df, 'prompt_tokens').mean()) if len(_num(df, 'prompt_tokens')) else NAN
    ref = ALL_JOBS.get(rec.get('reference_job_id'))
    tagged = df.copy()
    if ref is not None:
        rr = ref['rows'][ref['rows']['instance_id'].isin(set(df['instance_id']))]
        row.update({'gpu_' + k: v for k, v in job_metrics(rr).items()})
        rref = rr.drop_duplicates('instance_id').set_index('instance_id')
        for col in ('predicted_cypher', 'exact_match', 'correct', 'judged'):
            tagged['gpu_' + col] = tagged['instance_id'].map(rref[col])
    overall.append(row)
    for k in ('run_name', 'model', 'variant', 'condition', 'quant', 'prompt_schema_mode', 'train_schema_mode',
              'reference_job_id'):
        tagged[k] = rec.get(k)
    tagged['job_id'] = jid
    tagged['error_category'] = [_error_category(r) for r in tagged.to_dict('records')]
    long_frames.append(tagged)
    counts = tagged['error_category'].value_counts().to_dict()
    err_rows.append({'job_id': jid, **counts})

OVERALL = pd.DataFrame(overall)
EDGE_LONG = pd.concat(long_frames, ignore_index=True) if long_frames else pd.DataFrame()
ERRORS = pd.DataFrame(err_rows).fillna(0)
if not OVERALL.empty:
    OVERALL = OVERALL.sort_values(['model', 'variant', 'condition', 'quant']).reset_index(drop=True)
print('CPU jobs aggregated: ' + str(len(OVERALL)))


# --------------------------------------------------------------------------- #
# Paired comparisons on identical rows
# --------------------------------------------------------------------------- #
def edge_jid(run_name, condition, quant):
    for jid in EDGE_JOB_IDS:
        r = ALL_JOBS[jid]['job']
        if r.get('run_name') == run_name and r.get('condition') == condition and r.get('quant') == quant:
            return jid
    return None


def paired_compare(A, B, a_id, b_id, label, family):
    A = A.drop_duplicates('instance_id').set_index('instance_id')
    B = B.drop_duplicates('instance_id').set_index('instance_id')
    common = A.index.intersection(B.index)
    A, B = A.loc[common], B.loc[common]
    if not len(common):
        return []
    base = {'comparison': label, 'family': family, 'job_a': a_id, 'job_b': b_id,
            'identical_prediction_share': float((A['predicted_cypher'] == B['predicted_cypher']).mean()),
            'identical_normalised_share': float((A['predicted_cypher'].map(normalise_cypher)
                                                 == B['predicted_cypher'].map(normalise_cypher)).mean())}
    out = []
    for metric in ('exact_match', 'normalised_exact_match', 'execution_match'):
        if metric == 'execution_match':
            mask = (A['judged'] & B['judged']).to_numpy()
            a_ok, b_ok = A['correct'].to_numpy()[mask], B['correct'].to_numpy()[mask]
        else:
            a_ok, b_ok = A[metric].eq(True).to_numpy(), B[metric].eq(True).to_numpy()
        if not len(a_ok):
            continue
        stats = paired_diff_ci(a_ok, b_ok)
        stats.pop('bootstrap_p', None)
        a_only, b_only = int((a_ok & ~b_ok).sum()), int((b_ok & ~a_ok).sum())
        out.append({**base, 'metric': metric, 'n_pairs': len(a_ok), 'a_value': float(a_ok.mean()),
                    'b_value': float(b_ok.mean()), **stats, 'a_only_correct': a_only, 'b_only_correct': b_only,
                    'test': 'exact McNemar (two-sided)', 'p_value': mcnemar_exact(a_only, b_only),
                    'p_value_is_bound': False})
    ga = pd.to_numeric(A['gleu'], errors='coerce').fillna(0).to_numpy(float)
    gb = pd.to_numeric(B['gleu'], errors='coerce').fillna(0).to_numpy(float)
    stats = paired_diff_ci(ga, gb)
    p_raw = stats.pop('bootstrap_p')
    floor = 1.0 / BOOTSTRAP_SAMPLES
    out.append({**base, 'metric': 'sentence_gleu_mean', 'n_pairs': len(ga), 'a_value': float(ga.mean()),
                'b_value': float(gb.mean()), **stats, 'test': 'paired bootstrap (percentile)',
                'p_value': max(p_raw, floor), 'p_value_is_bound': p_raw < floor})
    for wall_col in ('edge_wall_ms', 'edge_prompt_ms'):
        if wall_col in A.columns and wall_col in B.columns:
            wa = pd.to_numeric(A[wall_col], errors='coerce')
            wb = pd.to_numeric(B[wall_col], errors='coerce')
            ok = (wa.notna() & wb.notna()).to_numpy()
            if ok.any():
                stats = paired_diff_ci(wa.to_numpy()[ok], wb.to_numpy()[ok])
                p_raw = stats.pop('bootstrap_p')
                for key in ('tost_ci_low', 'tost_ci_high', 'equivalent_within_margin'):
                    stats.pop(key, None)
                out.append({**base, 'metric': wall_col.replace('edge_', 'mean_'), 'n_pairs': int(ok.sum()),
                            'a_value': float(wa[ok].mean()), 'b_value': float(wb[ok].mean()), **stats,
                            'test': 'paired bootstrap (percentile)', 'p_value': max(p_raw, floor),
                            'p_value_is_bound': p_raw < floor})
    return out


COMPARISONS = []
for jid in EDGE_JOB_IDS:
    rec = ALL_JOBS[jid]['job']
    ref = ALL_JOBS.get(rec.get('reference_job_id'))
    if ref is not None:
        ids = set(ALL_JOBS[jid]['rows']['instance_id'])
        COMPARISONS.append((ref['rows'][ref['rows']['instance_id'].isin(ids)], ALL_JOBS[jid]['rows'],
                            rec['reference_job_id'], jid, jid + ': 4-bit HF (GPU) vs GGUF (CPU)', 'edge_vs_gpu'))
for e in EXP:
    for cond in ('fine_tuned', 'cross_schema'):
        a = edge_jid(e['run_name'], cond, ACCURACY_QUANT)
        for q in EXTRA_ACCURACY_QUANTS:
            b = edge_jid(e['run_name'], cond, q)
            if a and b:
                COMPARISONS.append((ALL_JOBS[a]['rows'], ALL_JOBS[b]['rows'], a, b,
                                    e['run_name'] + ' ' + cond + ': ' + ACCURACY_QUANT + ' vs ' + q, 'quant_fidelity'))
MODELS = list(dict.fromkeys(e['model'] for e in EXP))
for m in MODELS:
    enh, pru = BY_VARIANT.get((m, 'enhanced')), BY_VARIANT.get((m, 'pruned'))
    E = edge_jid(enh['run_name'], 'fine_tuned', ACCURACY_QUANT) if enh else None
    P = edge_jid(pru['run_name'], 'fine_tuned', ACCURACY_QUANT) if pru else None
    X = edge_jid(enh['run_name'], 'cross_schema', ACCURACY_QUANT) if enh else None
    for a, b, label, fam in ((E, P, m + ': enhanced vs exact-match (trained and prompted)', 'schema_filtering_edge'),
                             (E, X, m + ': enhanced-trained, enhanced vs exact-match prompts', 'schema_inference_effect_edge'),
                             (X, P, m + ': exact-match prompts, enhanced- vs exact-match-trained', 'schema_training_effect_edge')):
        if a and b:
            COMPARISONS.append((ALL_JOBS[a]['rows'], ALL_JOBS[b]['rows'], a, b, label, fam))
edge_enh = [(m, edge_jid(BY_VARIANT[(m, 'enhanced')]['run_name'], 'fine_tuned', ACCURACY_QUANT))
            for m in MODELS if (m, 'enhanced') in BY_VARIANT]
edge_enh = [x for x in edge_enh if x[1]]
for i, (ma, a) in enumerate(edge_enh):
    for mb, b in edge_enh[i + 1:]:
        COMPARISONS.append((ALL_JOBS[a]['rows'], ALL_JOBS[b]['rows'], a, b,
                            'CPU ' + ACCURACY_QUANT + ' enhanced: ' + ma + ' vs ' + mb, 'model_comparison_edge'))

paired_rows = []
for A, B, a, b, label, family in COMPARISONS:
    paired_rows.extend(paired_compare(A, B, a, b, label, family))
PAIRED = pd.DataFrame(paired_rows)
if not PAIRED.empty:
    PAIRED['p_holm'] = PAIRED.groupby(['family', 'metric'])['p_value'].transform(lambda s: holm(s.to_numpy()))
    PAIRED['significant_holm'] = PAIRED['p_holm'] < ALPHA
    floor_txt = '< ' + format(1.0 / BOOTSTRAP_SAMPLES, '.2g')
    PAIRED['p_value_display'] = [floor_txt if bound else format(p, '.3g')
                                 for p, bound in zip(PAIRED['p_value'], PAIRED['p_value_is_bound'])]

# --------------------------------------------------------------------------- #
# Fidelity summary, schema trade-off on the CPU, throughput, footprint, exports
# --------------------------------------------------------------------------- #
fid = []
if not PAIRED.empty:
    g = PAIRED[PAIRED['family'] == 'edge_vs_gpu']
    for jid, sub in g.groupby('job_b'):
        r = {'job_id': jid, 'reference_job_id': sub['job_a'].iloc[0],
             'identical_prediction_share': sub['identical_prediction_share'].iloc[0],
             'identical_normalised_share': sub['identical_normalised_share'].iloc[0]}
        for metric in ('exact_match', 'execution_match', 'sentence_gleu_mean'):
            m = sub[sub['metric'] == metric]
            if len(m):
                m = m.iloc[0]
                r.update({metric + '_gpu': m['a_value'], metric + '_cpu': m['b_value'],
                          metric + '_delta': m['difference_b_minus_a'], metric + '_delta_ci_low': m.get('diff_ci_low'),
                          metric + '_delta_ci_high': m.get('diff_ci_high'), metric + '_n': m['n_pairs'],
                          metric + '_p': m['p_value'], metric + '_p_holm': m['p_holm']})
                if metric == 'execution_match':
                    r['ea_equivalent_within_margin'] = m.get('equivalent_within_margin')
        fid.append(r)
FIDELITY = pd.DataFrame(fid)

trade = []
OV = OVERALL.set_index('job_id') if not OVERALL.empty else pd.DataFrame()
for m in MODELS:
    enh = BY_VARIANT.get((m, 'enhanced'))
    E = edge_jid(enh['run_name'], 'fine_tuned', ACCURACY_QUANT) if enh else None
    if not E:
        continue
    for label, other in (('exact_match_trained_and_prompted',
                          edge_jid(BY_VARIANT[(m, 'pruned')]['run_name'], 'fine_tuned', ACCURACY_QUANT)
                          if (m, 'pruned') in BY_VARIANT else None),
                         ('enhanced_trained_exact_match_prompts', edge_jid(enh['run_name'], 'cross_schema', ACCURACY_QUANT))):
        if not other:
            continue
        a, b = OV.loc[E], OV.loc[other]
        p = PAIRED[(PAIRED['job_a'] == E) & (PAIRED['job_b'] == other)] if not PAIRED.empty else PAIRED
        pe = p[p['metric'] == 'execution_match'].iloc[0] if len(p[p['metric'] == 'execution_match']) else None
        pw = p[p['metric'] == 'mean_wall_ms'].iloc[0] if len(p[p['metric'] == 'mean_wall_ms']) else None
        trade.append({
            'model': m, 'comparison': label, 'job_enhanced': E, 'job_filtered': other,
            'prompt_tokens_enhanced': _g(a, 'prompt_tokens_mean'), 'prompt_tokens_filtered': _g(b, 'prompt_tokens_mean'),
            'prefill_ms_p50_enhanced': _g(a, 'prefill_ms_p50'), 'prefill_ms_p50_filtered': _g(b, 'prefill_ms_p50'),
            'wall_ms_p50_enhanced': _g(a, 'wall_ms_p50'), 'wall_ms_p50_filtered': _g(b, 'wall_ms_p50'),
            'wall_ms_p95_enhanced': _g(a, 'wall_ms_p95'), 'wall_ms_p95_filtered': _g(b, 'wall_ms_p95'),
            'prefill_reduction_pct': 100 * (1 - _g(b, 'prefill_ms_mean') / _g(a, 'prefill_ms_mean'))
            if _g(a, 'prefill_ms_mean') else NAN,
            'wall_reduction_pct': 100 * (1 - _g(b, 'wall_ms_mean') / _g(a, 'wall_ms_mean')) if _g(a, 'wall_ms_mean') else NAN,
            'paired_wall_ms_delta': pw['difference_b_minus_a'] if pw is not None else NAN,
            'paired_wall_ms_delta_ci_low': pw['diff_ci_low'] if pw is not None else NAN,
            'paired_wall_ms_delta_ci_high': pw['diff_ci_high'] if pw is not None else NAN,
            'ea_enhanced': _g(a, 'execution_accuracy'), 'ea_filtered': _g(b, 'execution_accuracy'),
            'paired_ea_delta': pe['difference_b_minus_a'] if pe is not None else NAN,
            'paired_ea_delta_ci_low': pe['diff_ci_low'] if pe is not None else NAN,
            'paired_ea_delta_ci_high': pe['diff_ci_high'] if pe is not None else NAN,
            'paired_ea_n': int(pe['n_pairs']) if pe is not None else 0,
        })
SCHEMA_EDGE = pd.DataFrame(trade)

bench_rows = []
for model, rec in BENCH_RESULTS.items():
    for r in rec.get('bench_rows') or []:
        if r.get('error'):
            bench_rows.append({'model': model, 'quant': r.get('quant'), 'error': r['error']})
            continue
        size_b, params = r.get('model_size'), r.get('model_n_params')
        test = ('pp' + str(r.get('n_prompt'))) if r.get('n_prompt') and not r.get('n_gen') else ('tg' + str(r.get('n_gen')))
        bench_rows.append({'model': model, 'run_name': r.get('run_name'), 'quant': r.get('quant'), 'test': test,
                           'threads': r.get('n_threads'), 'tokens_per_s': r.get('avg_ts'),
                           'tokens_per_s_sd': r.get('stddev_ts'), 'model_size_mb': (size_b or 0) / 1e6 or NAN,
                           'model_n_params': params,
                           'bits_per_weight': 8 * size_b / params if size_b and params else NAN,
                           'gguf_size_mb': (r.get('gguf_size_bytes') or 0) / 1e6 or NAN,
                           'n_batch': r.get('n_batch'), 'n_ubatch': r.get('n_ubatch'), 'cpu_info': r.get('cpu_info'),
                           'build_commit': r.get('build_commit'), 'test_time': r.get('test_time')})
BENCH = pd.DataFrame(bench_rows)
FOOTPRINT = pd.DataFrame([{**fp, 'gguf_size_mb': (fp.get('gguf_size_bytes') or 0) / 1e6 or NAN}
                          for rec in BENCH_RESULTS.values() for fp in rec.get('footprint') or []])
THREADS_DF = pd.DataFrame([{'model': m, 'bench_run': rec.get('run_name'), 'status': rec.get('status'),
                            'threads_chosen': THREADS_FOR.get(m), 'thread_estimates_s': json.dumps(rec.get('thread_estimates_s')),
                            'typical_prompt_tokens': rec.get('typical_prompt_tokens'),
                            'typical_gen_tokens': rec.get('typical_gen_tokens'), 'est_s_per_item': rec.get('est_s_per_item')}
                           for m, rec in BENCH_RESULTS.items()])

exp_rows = []
for rn, rec in EXPORT_RECORDS.items():
    base = {'run_name': rn, 'model': rec.get('model'), 'status': rec.get('status'), 'merge_dtype': rec.get('merge_dtype'),
            'merge_s': rec.get('merge_s'), 'convert_s': rec.get('convert_s'),
            'merged_size_mb': (rec.get('merged_size_bytes') or 0) / 1e6 or NAN,
            'f16_size_mb': (rec.get('f16_size_bytes') or 0) / 1e6 or NAN,
            'adapter_verified': _flat(rec.get('adapter_verified')), 'adapter_remapped': _flat(rec.get('adapter_remapped')),
            'model_class': (rec.get('merge_metadata') or {}).get('model_class'),
            'llama_cpp_commit': rec.get('llama_cpp_commit'), 'session': rec.get('session'), 'error': rec.get('error')}
    if rec.get('quants'):
        for q, info in rec['quants'].items():
            exp_rows.append({**base, 'quant': q, 'file': info.get('file'), 'size_mb': info.get('size_bytes', 0) / 1e6,
                             'sha256': info.get('sha256'), 'quantize_s': info.get('quantize_s')})
    else:
        exp_rows.append(base)
EXPORTS_DF = pd.DataFrame(exp_rows)

parity = []
for jid in EDGE_JOB_IDS:
    df = ALL_JOBS[jid]['rows']
    srv = pd.to_numeric(df.get('edge_tokens_evaluated'), errors='coerce')
    hf = pd.to_numeric(df.get('prompt_tokens'), errors='coerce')
    diff = (srv - hf).dropna()
    parity.append({'job_id': jid, 'model': ALL_JOBS[jid]['job'].get('model'), 'n': len(diff),
                   'share_equal': float(diff.eq(0).mean()) if len(diff) else NAN,
                   'diff_counts': json.dumps({str(int(k)): int(v) for k, v in diff.value_counts().items()}),
                   'mean_diff': float(diff.mean()) if len(diff) else NAN})
TOKENIZER_PARITY = pd.DataFrame(parity)

# --------------------------------------------------------------------------- #
# Integrity checks
# --------------------------------------------------------------------------- #
INTEGRITY = []
INTEGRITY.append({'check': 'reference rows re-derived identically', 'subject': 'all',
                  'passed': not REF_ROWS_MISMATCH, 'detail': ', '.join(REF_ROWS_MISMATCH) or 'all match'})
INTEGRITY.append({'check': 'reference jobs from the current code commit', 'subject': 'all',
                  'passed': REF_COMMITS == [GIT_COMMIT], 'detail': ', '.join(c[:10] for c in REF_COMMITS)})
for jid in EDGE_JOB_IDS:
    ids = set(ALL_JOBS[jid]['rows']['instance_id'])
    INTEGRITY.append({'check': 'edge rows equal the planned subset', 'subject': jid,
                      'passed': ids == set(EDGE_IDS), 'detail': str(len(ids)) + ' rows'})
for rn, rec in EXPORT_RECORDS.items():
    if rec.get('status') == 'ok':
        INTEGRITY.append({'check': 'merged adapter verified (history/027.md)', 'subject': rn,
                          'passed': bool(rec.get('adapter_verified')), 'detail': _flat(rec.get('adapter_verified'))})
        INTEGRITY.append({'check': 'merge metadata records model_class', 'subject': rn,
                          'passed': bool((rec.get('merge_metadata') or {}).get('model_class')),
                          'detail': str((rec.get('merge_metadata') or {}).get('model_class'))})
for _, r in TOKENIZER_PARITY.iterrows():
    INTEGRITY.append({'check': 'GGUF prompt tokens equal training-tokenizer tokens (>=95% of rows)',
                      'subject': r['job_id'], 'passed': bool(r['share_equal'] >= 0.95) if r['share_equal'] == r['share_equal'] else None,
                      'detail': 'diff counts ' + r['diff_counts']})
for _, r in OVERALL.iterrows():
    INTEGRITY.append({'check': 'no llama-server request errors', 'subject': r['job_id'],
                      'passed': int(r.get('n_request_errors', 0)) == 0, 'detail': str(r.get('n_request_errors'))})
    INTEGRITY.append({'check': 'generation hit max_new_tokens in <5% of rows', 'subject': r['job_id'],
                      'passed': r.get('n_stopped_limit', 0) < 0.05 * r['n'], 'detail': str(r.get('n_stopped_limit'))})
    INTEGRITY.append({'check': 'no Neo4j connection failures', 'subject': r['job_id'],
                      'passed': int(r.get('n_connection_failed', 0)) == 0, 'detail': str(r.get('n_connection_failed'))})
recs = [ALL_JOBS[j]['job'] for j in EDGE_JOB_IDS] + [r for r in BENCH_RESULTS.values() if r.get('status') == 'completed']
for key, label in (('cpu_model', 'one CPU model across all edge measurements'),
                   ('llama_cpp_commit', 'one llama.cpp commit across all edge measurements'),
                   ('git_commit', 'one code commit across all edge measurements')):
    values = sorted({str(r.get(key)) for r in recs})
    INTEGRITY.append({'check': label, 'subject': 'all', 'passed': len(values) <= 1, 'detail': '; '.join(values)})
INTEGRITY.append({'check': 'code version pinned with GIT_REF', 'subject': 'all', 'passed': bool(GIT_REF),
                  'detail': GIT_REF or '(blank)'})
INTEGRITY.append({'check': 'llama.cpp pinned (LLAMA_CPP_REF or Hub pin)', 'subject': 'all',
                  'passed': bool(LLAMA_REF_USED), 'detail': str(LLAMA_REF_USED or 'latest at session start')})
for job in JOBS:
    status = JOB_RESULTS.get(job['job_id'], {}).get('status', 'not_started')
    INTEGRITY.append({'check': 'expected job completed', 'subject': job['job_id'], 'passed': status == 'completed',
                      'detail': status + ((': ' + str(JOB_RESULTS[job['job_id']].get('error'))[:200])
                                          if JOB_RESULTS.get(job['job_id'], {}).get('error') else '')})
for model, rec in BENCH_RESULTS.items():
    INTEGRITY.append({'check': 'throughput benchmark completed', 'subject': model,
                      'passed': rec.get('status') == 'completed', 'detail': str(rec.get('status'))})
INTEGRITY_DF = pd.DataFrame(INTEGRITY)


# --------------------------------------------------------------------------- #
# Headline
# --------------------------------------------------------------------------- #
def _ci(row, metric):
    v = _g(row, metric)
    if v != v:
        return ''
    lo, hi = _g(row, metric + '_ci_low'), _g(row, metric + '_ci_high')
    return format(v, '.3f') if lo != lo else format(v, '.3f') + ' [' + format(lo, '.3f') + ', ' + format(hi, '.3f') + ']'


def _fmt(v, spec='.3f'):
    return '' if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v) else format(float(v), spec)


FID = FIDELITY.set_index('job_id') if not FIDELITY.empty else pd.DataFrame()
COND_LABELS = {'fine_tuned': 'fine-tuned', 'cross_schema': 'fine-tuned (trained enhanced, exact-match prompts)'}
HEADLINE = pd.DataFrame([{
    'Model': r['model'], 'Schema': r['prompt_schema_mode'], 'Condition': COND_LABELS.get(r['condition'], r['condition']),
    'Quant': r['quant'], 'N': int(r['n']),
    'Exec. acc. CPU': _ci(r, 'execution_accuracy'), 'Exec. acc. GPU (same rows)': _fmt(r.get('gpu_execution_accuracy')),
    'Delta EA [CI]': (_fmt(FID.loc[r['job_id'], 'execution_match_delta']) + ' [' + _fmt(FID.loc[r['job_id'], 'execution_match_delta_ci_low'])
                      + ', ' + _fmt(FID.loc[r['job_id'], 'execution_match_delta_ci_high']) + ']')
    if r['job_id'] in FID.index and 'execution_match_delta' in FID.columns else '',
    'Exec. N': _fmt(r.get('n_judged'), '.0f'), 'EM CPU': _fmt(r.get('exact_match')),
    'EM GPU': _fmt(r.get('gpu_exact_match')), 'GLEU CPU': _fmt(r.get('google_bleu_corpus')),
    'Identical to GPU': _fmt(FID.loc[r['job_id'], 'identical_normalised_share'], '.1%')
    if r['job_id'] in FID.index else '',
    'Median s/query': _fmt(_g(r, 'wall_ms_p50') / 1000, '.1f'), 'p95 s/query': _fmt(_g(r, 'wall_ms_p95') / 1000, '.1f'),
    'Prefill share': _fmt(r.get('prefill_share_of_wall'), '.0%'),
    'Prompt tokens': _fmt(r.get('prompt_tokens_mean'), '.0f'), 'GGUF MB': _fmt(r.get('gguf_size_mb'), '.0f'),
    'Peak RSS MB': _fmt(r.get('peak_rss_mb'), '.0f'), 'Threads': _fmt(r.get('threads'), '.0f'),
} for _, r in OVERALL.iterrows()]) if not OVERALL.empty else pd.DataFrame()

print('Edge headline (' + str(int(CI_LEVEL * 100)) + '% CI in brackets):')
display(HEADLINE)
if not SCHEMA_EDGE.empty:
    print('Schema filtering on the CPU:')
    display(SCHEMA_EDGE)
failed_checks = INTEGRITY_DF[INTEGRITY_DF['passed'].eq(False)]
print('Integrity checks: ' + str(len(INTEGRITY_DF)) + ' run, ' + str(len(failed_checks)) + ' failed')
if len(failed_checks):
    display(failed_checks)

In [ ]:
# 13. Figures for the thesis (PNG at 300 dpi + vector PDF)
import matplotlib.pyplot as plt

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 9})


def save_fig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(FIG_DIR / (name + '.' + ext), bbox_inches='tight')
    plt.show()
    plt.close(fig)


def _short(jid):
    return jid.replace('__fine_tuned', '').replace('__gguf_', LF).replace('_cpu', '').replace('_prompts', ' prompts')


# Figure 1: execution accuracy, GPU 4-bit vs CPU GGUF on identical rows (Wilson CIs).
if not OVERALL.empty and OVERALL['execution_accuracy'].notna().any():
    sub = OVERALL.dropna(subset=['execution_accuracy'])
    x = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(max(6, 1.3 * len(sub)), 4))
    for k, (prefix, label) in enumerate((('gpu_', 'GPU 4-bit HF (same rows)'), ('', 'CPU GGUF'))):
        col = prefix + 'execution_accuracy'
        if col not in sub.columns:
            continue
        vals = sub[col].to_numpy(float)
        lo = sub.get(col + '_ci_low', sub[col]).to_numpy(float)
        hi = sub.get(col + '_ci_high', sub[col]).to_numpy(float)
        yerr = np.clip(np.nan_to_num(np.vstack([vals - lo, hi - vals])), 0, None)
        ax.bar(x + (k - 0.5) * 0.38, np.nan_to_num(vals), 0.38, yerr=yerr, capsize=3, label=label)
    ax.set_xticks(x)
    ax.set_xticklabels([_short(j) for j in sub['job_id']], fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Execution accuracy')
    ax.set_title('Execution accuracy on identical rows: GPU vs CPU (' + str(int(CI_LEVEL * 100)) + '% CI)')
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    save_fig(fig, 'fig1_edge_vs_gpu_execution_accuracy')

# Figure 2: per-query latency, median prefill + decode, with p95 marker.
if not OVERALL.empty and 'prefill_ms_p50' in OVERALL.columns:
    sub = OVERALL.dropna(subset=['wall_ms_p50'])
    y = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(sub) + 1.5))
    pre = sub['prefill_ms_p50'].to_numpy(float) / 1000
    dec = sub['decode_ms_p50'].to_numpy(float) / 1000
    ax.barh(y, pre, label='prefill (median)')
    ax.barh(y, dec, left=pre, label='decode (median)')
    ax.scatter(sub['wall_ms_p95'].to_numpy(float) / 1000, y, marker='|', s=200, color='black', label='p95 total')
    ax.set_yticks(y)
    ax.set_yticklabels([_short(j).replace(LF, ' ') for j in sub['job_id']], fontsize=7)
    ax.set_xlabel('Seconds per query (' + CPU_MODEL[:40] + ')')
    ax.set_title('Per-query latency on the CPU')
    ax.grid(axis='x', alpha=0.3)
    ax.legend(fontsize=7)
    save_fig(fig, 'fig2_latency_breakdown')

# Figure 3: llama-bench throughput vs threads.
if not BENCH.empty and 'tokens_per_s' in BENCH.columns:
    sweep = BENCH[BENCH['quant'].isin(THREAD_SWEEP_QUANTS) & BENCH['tokens_per_s'].notna()]
    if not sweep.empty:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for ax, kind in zip(axes, ('pp', 'tg')):
            for (model, quant), g in sweep[sweep['test'].str.startswith(kind)].groupby(['model', 'quant']):
                g = g.sort_values('threads')
                ax.errorbar(g['threads'], g['tokens_per_s'], yerr=g['tokens_per_s_sd'], marker='o', capsize=3,
                            label=model + ' ' + quant)
            ax.set_xlabel('Threads')
            ax.set_ylabel('Tokens/s')
            ax.set_title(('Prompt processing (pp' + str(BENCH_PROMPT_TOKENS) + ')') if kind == 'pp'
                         else ('Generation (tg' + str(BENCH_GEN_TOKENS) + ')'))
            ax.grid(alpha=0.3)
            ax.legend(fontsize=7)
        save_fig(fig, 'fig3_throughput_vs_threads')

# Figure 4: quantisation trade-off: file size vs generation speed (full thread count).
if not BENCH.empty and 'tokens_per_s' in BENCH.columns:
    tg = BENCH[BENCH['test'].str.startswith('tg') & BENCH['threads'].eq(LOGICAL_CPUS)].dropna(subset=['tokens_per_s'])
    if not tg.empty:
        fig, ax = plt.subplots(figsize=(7, 4))
        for model, g in tg.groupby('model'):
            ax.plot(g['gguf_size_mb'], g['tokens_per_s'], marker='o', label=model)
            for _, r in g.iterrows():
                ax.annotate(r['quant'], (r['gguf_size_mb'], r['tokens_per_s']), fontsize=7,
                            xytext=(3, 3), textcoords='offset points')
        ax.set_xlabel('GGUF size (MB)')
        ax.set_ylabel('Generation tokens/s (' + str(LOGICAL_CPUS) + ' threads)')
        ax.set_title('Quantisation: size vs generation speed')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
        save_fig(fig, 'fig4_quant_size_vs_speed')

# Figure 5: prefill time vs prompt tokens (why schema filtering matters on the CPU).
if not EDGE_LONG.empty and 'edge_prompt_ms' in EDGE_LONG.columns:
    pts = EDGE_LONG[EDGE_LONG['quant'] == ACCURACY_QUANT]
    if not pts.empty:
        fig, ax = plt.subplots(figsize=(7, 4))
        for (model, mode), g in pts.groupby(['model', 'prompt_schema_mode']):
            ax.scatter(pd.to_numeric(g['edge_tokens_evaluated'], errors='coerce'),
                       pd.to_numeric(g['edge_prompt_ms'], errors='coerce') / 1000, s=8, alpha=0.6,
                       label=model + ' / ' + mode)
        ax.set_xlabel('Prompt tokens')
        ax.set_ylabel('Prefill seconds')
        ax.set_title('Prefill cost vs prompt length (' + ACCURACY_QUANT + ')')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
        save_fig(fig, 'fig5_prefill_vs_prompt_tokens')

# Figure 6: exact-match filter coverage of gold-query elements by data source.
if not COVERAGE.empty:
    sub = COVERAGE.set_index('group').sort_values('n_parsed', ascending=False)
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(sub) + 1.5))
    sub[['share_fully_covered', 'share_missing_label', 'share_missing_rel_type', 'share_missing_property']].plot(
        kind='barh', ax=ax, width=0.8)
    ax.set_xlim(0, 1)
    ax.set_yticklabels([g + ' (n=' + str(int(n)) + ')' for g, n in zip(sub.index, sub['n_parsed'])], fontsize=7)
    ax.set_xlabel('Share of rows')
    ax.set_title('Exact-match filter: gold-query elements kept in the prompt schema')
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
    save_fig(fig, 'fig6_filter_coverage')

# Figure 7: schema hallucination per job.
if not HALLUCINATION.empty:
    sub = HALLUCINATION.sort_values(['source', 'job_id'])
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(sub) + 1.5))
    sub.set_index('job_id')[['share_unknown_label', 'share_unknown_rel_type', 'share_unknown_property']].plot(
        kind='barh', ax=ax, width=0.8)
    ax.set_xlabel('Share of rows with a non-empty prediction')
    ax.set_title('Predicted schema elements absent from the full schema')
    ax.tick_params(axis='y', labelsize=6)
    ax.legend(fontsize=7)
    save_fig(fig, 'fig7_schema_hallucination')

# Figure 8: accuracy vs memory footprint on the CPU.
if not OVERALL.empty and OVERALL['peak_rss_mb'].notna().any() and OVERALL['execution_accuracy'].notna().any():
    fig, ax = plt.subplots(figsize=(7, 4))
    for _, r in OVERALL.dropna(subset=['peak_rss_mb', 'execution_accuracy']).iterrows():
        ax.scatter(r['peak_rss_mb'], r['execution_accuracy'])
        ax.annotate(_short(r['job_id']).replace(LF, ' '), (r['peak_rss_mb'], r['execution_accuracy']), fontsize=6,
                    xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel('Peak resident memory of llama-server (MB)')
    ax.set_ylabel('Execution accuracy (CPU)')
    ax.set_title('Accuracy vs memory footprint')
    ax.grid(alpha=0.3)
    save_fig(fig, 'fig8_accuracy_vs_memory')

print('Figures saved to ' + str(FIG_DIR))

In [ ]:
# 14. Write the structured results bundle, upload it, save it with the version and finish
import importlib.metadata as importlib_metadata


def _records(df):
    if df is None or df.empty:
        return []
    return json.loads(df.to_json(orient='records', date_format='iso', default_handler=str))


def _version(dist):
    try:
        return importlib_metadata.version(dist)
    except Exception:
        return None


created = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
MANIFEST = {
    'title': 'Text2Cypher LoRA fine-tuning: edge (CPU) evaluation and supplementary analyses',
    'created_utc': created, 'session': SESSION, 'edge_tag': EDGE_TAG, 'edge_prefix': EDGE_PREFIX,
    'reference_eval_tag': REFERENCE_EVAL_TAG, 'reference_commits': REF_COMMITS,
    'split': SPLIT, 'reference_limit': REFERENCE_LIMIT, 'edge_limit': len(EDGE_IDS), 'edge_subset': EDGE_SUBSET,
    'edge_subset_seed': EDGE_SUBSET_SEED, 'sample_seed': SAMPLE_SEED, 'edge_ids_sha': EDGE_IDS_SHA,
    'edge_rows_with_database': n_db_edge,
    'git_repo': GITHUB_REPO, 'git_ref': GIT_REF, 'git_commit': GIT_COMMIT, 'git_dirty': GIT_DIRTY,
    'llama_cpp_repo': LLAMA_CPP_REPO, 'llama_cpp_ref': LLAMA_REF_USED, 'llama_cpp_commit': LLAMA_COMMIT,
    'llama_cpp_version': LLAMA_VERSION,
    'llama_cpp_build': 'CPU only, GGML_NATIVE=ON (kernels tuned to the host CPU), Release',
    'hub_repo': HUB_REPO, 'platform_name': PLATFORM, 'kaggle_run_type': RUN_TYPE,
    'cpu': {'model': CPU_MODEL, 'logical_cpus': LOGICAL_CPUS, 'physical_cores': PHYSICAL_CORES,
            'simd_flags': SIMD_FLAGS, 'cgroup_cpu_limit': CGROUP_CPU_LIMIT, 'ram_total_gb': round(RAM_TOTAL_GB, 1),
            'lscpu': {k: LSCPU.get(k) for k in ('Architecture', 'Model name', 'Socket(s)', 'Core(s) per socket',
                                                'Thread(s) per core', 'L2 cache', 'L3 cache', 'Hypervisor vendor')}},
    'gpu_attached_but_unused': GPUS_PRESENT,
    'python': platform.python_version(), 'platform': platform.platform(),
    'packages': {d: _version(d) for d in ('torch', 'transformers', 'peft', 'safetensors', 'huggingface_hub', 'gguf',
                                          'neo4j', 'pandas', 'numpy', 'requests')},
    'export': {'merge_dtype': MERGE_DTYPE, 'intermediate': 'f16 GGUF (convert_hf_to_gguf.py)',
               'quantiser': 'llama-quantize from the same build'},
    'serving': {'server': 'llama-server', 'flags': '--parallel 1 --no-mmap --no-warmup', 'threads': THREADS_FOR,
                'server_threads_setting': SERVER_THREADS, 'cache_prompt': CACHE_PROMPT, 'edge_stop_mode': EDGE_STOP_MODE,
                'decoding': 'greedy (temperature 0, top_k 1); max_new_tokens from each config',
                'warmup': 'one untimed 1-token request after load, excluded from latency statistics'},
    'benchmark': {'tool': 'llama-bench', 'prompt_tokens': BENCH_PROMPT_TOKENS, 'gen_tokens': BENCH_GEN_TOKENS,
                  'reps': BENCH_REPS, 'threads': BENCH_THREADS, 'thread_sweep_quants': THREAD_SWEEP_QUANTS,
                  'latency_quants': LATENCY_QUANTS},
    'execution_provider': 'Neo4j Labs demo server (neo4j+s://demo.neo4jlabs.com:7687)',
    'dead_db_policy': DEAD_DB_POLICY,
    'statistics': {'ci_level': CI_LEVEL, 'bootstrap_samples': BOOTSTRAP_SAMPLES, 'stats_seed': STATS_SEED,
                   'proportion_ci': 'Wilson score interval', 'gleu_ci': 'percentile bootstrap over rows',
                   'paired_tests': 'exact McNemar for binary metrics; paired bootstrap for mean sentence GLEU and '
                                   'latency; Holm correction within each family x metric',
                   'equivalence': 'TOST via the (1 - 2 alpha) paired bootstrap CI, margin ' + str(EQUIV_MARGIN)},
    'experiments': [{k: _flat(e.get(k)) for k in ('model', 'variant', 'config', 'run_name', 'schema_mode', 'adapter_ok',
                                                  'fingerprint_trained', 'fingerprint_now', 'fingerprint_match')}
                    for e in EXP],
    'jobs': {jid: rec.get('status') for jid, rec in JOB_RESULTS.items()},
    'jobs_expected': [j['job_id'] for j in JOBS],
    'jobs_not_run': [j['job_id'] for j in JOBS if JOB_RESULTS.get(j['job_id'], {}).get('status') != 'completed'],
    'jobs_skipped_for_time': SKIPPED_FOR_TIME,
    'bench': {m: rec.get('status') for m, rec in BENCH_RESULTS.items()},
    'exports': {rn: rec.get('status') for rn, rec in EXPORT_RECORDS.items()},
    'supplementary': RUN_SUPPLEMENTARY, 'filter_mode': FILTER_MODE, 'filter_params': FILTER_KW,
}

TABLES = {
    'edge_headline': HEADLINE, 'edge_overall_metrics': OVERALL, 'edge_fidelity_vs_gpu': FIDELITY,
    'edge_paired_tests': PAIRED, 'edge_schema_tradeoff': SCHEMA_EDGE, 'edge_error_categories': ERRORS,
    'edge_tokenizer_parity': TOKENIZER_PARITY, 'throughput_llama_bench': BENCH, 'footprint': FOOTPRINT,
    'serving_threads': THREADS_DF, 'exports': EXPORTS_DF, 'integrity_checks': INTEGRITY_DF,
    'jobs': pd.DataFrame([{k: _flat(v) for k, v in rec.items()} for rec in JOB_RESULTS.values()]),
    'supp_subset_metrics': SUBSET_METRICS, 'supp_filter_coverage': COVERAGE,
    'supp_filter_coverage_rows': COVERAGE_ROWS, 'supp_filter_coverage_vs_ea': COVERAGE_EA,
    'supp_schema_hallucination': HALLUCINATION, 'supp_execution_result_breakdown': RESULT_BREAKDOWN,
    'supp_model_agreement': AGREEMENT, 'supp_sample_composition': COMPOSITION,
}
for name, df in TABLES.items():
    if df is not None:
        df.to_csv(OUT_DIR / 'tables' / (name + '.csv'), index=False)
if not EDGE_LONG.empty:
    try:
        EDGE_LONG.to_parquet(OUT_DIR / 'edge_predictions_long.parquet', index=False)
    except Exception as exc:
        print('Parquet export failed (' + str(exc) + '); the gzipped CSV is still written.')
    EDGE_LONG.drop(columns=['predicted_raw'], errors='ignore').to_csv(
        OUT_DIR / 'edge_predictions_long.csv.gz', index=False, compression='gzip')
for name, df in (('edge_headline', HEADLINE), ('edge_schema_tradeoff', SCHEMA_EDGE)):
    try:
        if not df.empty:
            df.to_latex(OUT_DIR / 'tables' / (name + '.tex'), index=False)
    except Exception as exc:
        print('LaTeX export of ' + name + ' skipped: ' + str(exc))

results = {'manifest': MANIFEST}
results.update({name: _records(df) for name, df in TABLES.items() if name != 'supp_filter_coverage_rows'})
(OUT_DIR / 'results.json').write_text(json.dumps(results, indent=2, default=str), encoding='utf-8')
(OUT_DIR / 'manifest.json').write_text(json.dumps(MANIFEST, indent=2, default=str), encoding='utf-8')

# Per-job logs, reports and records; bench and export records.
for sub_name, src_root, names in (('jobs', EDGE_DIR / 'jobs', ('job.json', 'report.md', 'evaluate.log', 'server.log')),
                                  ('bench', EDGE_DIR / 'bench', ('bench.json',)),
                                  ('exports', EDGE_DIR / 'exports', None)):
    if not src_root.exists():
        continue
    for item in sorted(src_root.iterdir()):
        dest = OUT_DIR / sub_name
        dest.mkdir(exist_ok=True)
        if item.is_dir():
            for fname in names or ():
                if (item / fname).exists():
                    (dest / item.name).mkdir(exist_ok=True)
                    shutil.copy2(item / fname, dest / item.name / fname)
        elif names is None:
            shutil.copy2(item, dest / item.name)

cpu_short = CPU_MODEL + ', ' + str(LOGICAL_CPUS) + ' logical CPUs'
readme = [
    '# Text2Cypher edge (CPU) evaluation: ' + EDGE_TAG,
    '',
    'Generated ' + created + ' on Kaggle (session ' + SESSION + ', commit ' + GIT_COMMIT[:10] + ', llama.cpp '
    + LLAMA_COMMIT[:10] + ', ' + cpu_short + ').',
    'GPU reference: `final-eval/' + REFERENCE_EVAL_TAG + '` (commits ' + ', '.join(c[:10] for c in REF_COMMITS) + ').',
    '',
    '## Files',
    '- `tables/edge_headline.csv` (+ `.tex`): one row per CPU job: accuracy with CIs, GPU accuracy on the same rows, '
    'paired delta, share of identical predictions, latency, size and memory.',
    '- `tables/edge_overall_metrics.csv`: all metrics, CI bounds, latency percentiles and provenance per CPU job; '
    '`gpu_*` columns are the reference job restricted to the same rows.',
    '- `tables/edge_fidelity_vs_gpu.csv`: CPU vs GPU per job (EM, execution match, sentence GLEU deltas with CIs, '
    'TOST equivalence, identical-prediction shares).',
    '- `tables/edge_paired_tests.csv`: every paired comparison (families edge_vs_gpu, quant_fidelity, '
    'schema_filtering_edge, schema_inference_effect_edge, schema_training_effect_edge, model_comparison_edge).',
    '- `tables/edge_schema_tradeoff.csv`: enhanced vs exact-match schema on the CPU (prompt tokens, prefill and total '
    'latency vs execution accuracy).',
    '- `tables/throughput_llama_bench.csv`: llama-bench tokens/s per model, quantisation, test (ppN/tgN) and threads, '
    'with bits per weight.',
    '- `tables/footprint.csv`: llama-server load time, RSS after load, peak RSS and first-request time per quantisation.',
    '- `tables/serving_threads.csv`: thread count used for serving and the estimates it was chosen from.',
    '- `tables/exports.csv`: merge/convert/quantise timings, sizes, SHA-256, adapter verification and model class.',
    '- `tables/edge_tokenizer_parity.csv`: GGUF prompt tokens minus training-tokenizer tokens per job.',
    '- `tables/edge_error_categories.csv`, `tables/integrity_checks.csv`, `tables/jobs.csv`.',
    '- `tables/supp_*.csv`: supplementary analyses of GPU reference and CPU jobs (see below).',
    '- `edge_predictions_long.parquet` / `.csv.gz`: one row per CPU job and instance, with server timings and the '
    'matching GPU prediction and correctness.',
    '- `results.json` (all tables except per-row coverage, plus the manifest), `manifest.json`, `figures/`, '
    '`jobs/`, `bench/`, `exports/`.',
    '',
    '## Metric definitions',
    '- `execution_accuracy`: share of judged rows (a demo database, an executable gold query, no outage or excluded '
    'database) whose result set equals the gold result. Wilson CI. `execution_accuracy_nonempty_gold` restricts it to '
    'rows whose gold query returns at least one row; `share_correct_with_empty_gold` is the share of matches where the '
    'gold result is empty.',
    '- `exact_match`, `normalised_exact_match`, `google_bleu_corpus`: as in final-eval (Wilson CIs; percentile '
    'bootstrap for GLEU).',
    '- Latency: `wall_ms` is client-measured time per request (one request at a time, after one untimed warm-up); '
    '`prefill_ms` and `decode_ms` are llama-server timings for prompt processing and generation.',
    '- `prefill_share_of_wall`: summed prefill time over summed wall time. `queries_per_hour`: 3.6e6 / mean wall ms.',
    '- Footprint: `load_s` is process start to a healthy /health endpoint with `--no-mmap` (weights read into memory); '
    '`peak_rss_mb` is VmHWM of the server process.',
    '- Throughput: llama-bench `avg_ts` (tokens/s) for ppN (prompt processing) and tgN (generation), ' + str(BENCH_REPS)
    + ' repetitions. `bits_per_weight` = 8 * model_size / model_n_params as reported by llama-bench.',
    '- Tokenizer parity: `tokens_evaluated` reported by llama-server minus `prompt_tokens` from the training tokenizer. '
    '0 means the GGUF model saw the same token count as in training (BOS handled identically).',
    '- Paired tests: exact McNemar for binary metrics on identical rows; paired bootstrap for sentence GLEU and latency; '
    'Holm correction within each family x metric. TOST equivalence at +/-' + str(EQUIV_MARGIN) + ' for accuracy deltas.',
    '',
    '## Supplementary analyses',
    '- `supp_subset_metrics`: metrics on all rows, without `' + FUNCTIONAL_SOURCE + '` (near-saturated EM inflates '
    'text metrics), with and without a database.',
    '- `supp_filter_coverage*`: whether the ' + FILTER_MODE + ' filter keeps every label, relationship type and '
    'property the gold query uses (heuristic extraction; name-level for properties; label predicates in WHERE are not '
    'parsed). `supp_filter_coverage_vs_ea` splits execution accuracy by fully covered vs not.',
    '- `supp_schema_hallucination`: predicted labels/relationship types/properties absent from the full schema, and use '
    'of real elements that were not in the prompt schema (prompt schema computed before budget trimming).',
    '- `supp_execution_result_breakdown`: judged rows split into correct (non-empty / empty gold), prediction failed, '
    'empty prediction result, row-count mismatch and same row count with different values.',
    '- `supp_model_agreement`: oracle, majority and all-correct execution accuracy across the fine-tuned enhanced GPU '
    'models on rows judged for all of them; pairwise identical-prediction shares.',
    '- `supp_sample_composition`: data source x database access x complexity counts for the 500 reference rows and the '
    'edge subset.',
    '',
    '## Caveats',
    '- CPU jobs score ' + str(len(EDGE_IDS)) + ' of the ' + str(REFERENCE_LIMIT) + ' reference rows (' + EDGE_SUBSET
    + ', seed ' + str(EDGE_SUBSET_SEED) + '); only ' + str(n_db_edge) + ' have a database, so CPU execution-accuracy '
    'CIs are wide. GPU comparisons always use the same rows.',
    '- Timings are for ' + cpu_short + ' (Kaggle VM), not a specific edge device; the thread sweep indicates scaling '
    'to fewer cores. The llama.cpp build is tuned to this CPU (GGML_NATIVE).',
    '- Adapters were trained against an NF4-quantised base and merged into the ' + MERGE_DTYPE + ' base before GGUF '
    'quantisation; CPU-vs-GPU differences combine the merge and the GGUF quantisation.',
    '- Stop condition: ' + ('EOS only, matching the GPU jobs' if EDGE_STOP_MODE == 'hf'
                            else 'EOS plus generation.stop_strings (differs from the GPU jobs)') + '.',
    '- Prompt caching was ' + ('on' if CACHE_PROMPT else 'off') + ': each query pays the full prefill.',
    '- Load times may benefit from the OS page cache because GGUFs were written shortly before loading.',
    '- Element extraction for coverage and hallucination is regex-based and approximate.',
]
(OUT_DIR / 'README.md').write_text(LF.join(readme) + LF, encoding='utf-8')

SAVED_DIR = Path(OUTPUT_DIR) / 'edge-results' / EDGE_TAG / SESSION
shutil.copytree(OUT_DIR, SAVED_DIR, dirs_exist_ok=True)
archive = shutil.make_archive(str(Path(OUTPUT_DIR) / ('edge-results-' + EDGE_TAG + '-' + SESSION)), 'zip',
                              root_dir=OUT_DIR)
uploaded = hub_upload_folder(OUT_DIR, EDGE_PREFIX + '/thesis/' + SESSION,
                             '[edge-eval] thesis bundle ' + EDGE_TAG + ' ' + SESSION)
print('Results bundle : ' + str(SAVED_DIR) + ' (saved with the version)')
print('Zip archive    : ' + archive)
print('Logs           : ' + str(LOG_DIR))
if UPLOAD_TO_HUB:
    print('Hub upload     : ' + ('OK' if uploaded else 'FAILED') + ' (https://huggingface.co/' + HUB_REPO
          + '/tree/main/' + EDGE_PREFIX + '/thesis/' + SESSION + ')')

summary = ['Session ' + SESSION + ' finished (' + EDGE_TAG + ', ' + CPU_MODEL[:40] + ').']
for _, r in OVERALL.iterrows():
    summary.append(str(r['job_id']) + ': EA ' + _fmt(r.get('execution_accuracy')) + ' (GPU '
                   + _fmt(r.get('gpu_execution_accuracy')) + '), median ' + _fmt(_g(r, 'wall_ms_p50') / 1000, '.1f')
                   + ' s/query, peak ' + _fmt(r.get('peak_rss_mb'), '.0f') + ' MB')
if MANIFEST['jobs_not_run']:
    summary.append('Not run: ' + ', '.join(MANIFEST['jobs_not_run']) + '. Commit a new version to finish them.')
if len(failed_checks):
    summary.append(str(len(failed_checks)) + ' integrity check(s) failed; see integrity_checks.csv.')
STATUS['outcome'] = 'completed'
notify('finished', LF.join(summary),
       tags=['warning'] if len(failed_checks) or MANIFEST['jobs_not_run'] else ['white_check_mark'])
display(HEADLINE)